# Tunex
## Overall Training and Evaluation Strategy

### High-level approach

The primary goal of this project was not raw benchmark accuracy, but reliable, judge-visible reasoning that is explicit, deterministic, structurally correct, and reproducible without inference-time fixes.

From the start, output format was treated as a first-class learning objective. Any approach relying on hidden chain-of-thought, prompt tricks, or post-processing was intentionally avoided, because these methods can hide failure modes and diverge from judge behavior.

The model is directly optimized to emit the required XML-style structure, evaluated using judge-equivalent decoding, and allowed to fail visibly if it produces invalid output.

---

### Compute allocation and training setup

All training is performed in a single Kaggle TPU session to maximize reproducibility and reduce infrastructure complexity.

Key choices:

* Base model: google/gemma-3-1b-it (publicly available)
* Framework: Tunix with JAX and Flax NNX
* Training method: GRPO (policy-only reinforcement learning, no critic)
* TPU mesh: fsdp=1, tp=4
* Batch size: 8
* Total training steps: 600

Compute is deliberately allocated toward short, high-signal training runs with aggressive early constraints and frequent diagnostics, rather than long unconstrained optimization.

---

### Reward design philosophy

The reward function is structure-first, quality-second, correctness-last.

This ordering is intentional:

1. A correct answer in the wrong format is unusable for evaluation
2. Clear and explicit reasoning matters more than task-specific accuracy
3. Correctness bonuses are meaningful only after structural stability

The reward function is deterministic and gate-based. It explicitly penalizes:

* Missing or duplicated tags
* Incorrect tag ordering
* Multiple reasoning blocks
* Text appearing after the closing answer tag
* Control tokens and chat artifacts
* Banned phrases such as “Question:” or “Final Answer:”
* The specific failure mode where the answer is closed using a reasoning tag

A critical fix was moving truncation to occur only after full structural validation. Earlier versions truncated output before validation, which masked structural errors and prevented the model from learning to fix them.

Partial rewards are used only in early training stages to stabilize learning.

---

### Staged GRPO curriculum

Training is divided into three explicit stages.

Stage 1: Format Lock
The goal is to make the output structure non-negotiable. Partial credit is allowed for progress toward correct structure, while control tokens are heavily penalized.

Stage 2: Quality Enforcement
Exactly one reasoning and one answer block are required. Minimum reasoning length is enforced, and stricter ordering checks are applied, with soft penalties allowing recovery.

Stage 3: Strict Mode
Only fully valid outputs are accepted. Any extra text, duplicated reasoning, banned patterns, or structural violations receive strong negative rewards.

This staged approach avoids early collapse while converging to strict correctness.

---

### Training pipeline and diagnostics

The training loop is fully instrumented for transparency.

A custom RewardDebugger tracks:

* Gate-level failure statistics
* Reward distribution
* Sample-level failures and successes
* Training progress over time

This makes it possible to identify dominant failure modes, verify that fixes change behavior, and catch regressions during phase transitions.

Checkpoints are saved using Orbax, and the reference model is explicitly synchronized between phases to stabilize KL behavior.

---

### Evaluation methodology

Evaluation is intentionally strict and judge-equivalent.

* Rollout engine: VanillaRollout
* Decoding: greedy
* Temperature: 0.0
* Top-K: 1
* Top-P: disabled

Because the base model is chat-tuned, the end-of-turn token can leak during inference. To prevent this while remaining faithful to judge behavior, evaluation treats both tokens as end-of-sequence:

* tokenizer.eos_id()
* tokenizer.piece_to_id("<end_of_turn>")

No output sanitization or post-processing is applied. Invalid outputs are surfaced exactly as generated.

---

### Ablations and lessons learned

Several lessons emerged during development:

* Without strong early format enforcement, the model collapses into template copying
* Truncating output before validation prevents the model from learning structural errors
* High KL pressure too early degrades reasoning quality
* Staged constraints are more stable than uniform enforcement

The strongest conclusion is that format must be enforced aggressively from the very beginning, and reasoning quality should be optimized only after structure is locked.

---

### Summary

This project demonstrates that structured, transparent reasoning can be trained reliably without hidden chain-of-thought or inference-time fixes. GRPO with staged constraints is effective and stable, and Tunix enables fine-grained reward control in a fully reproducible Kaggle TPU setup.

The overall strategy prioritizes clarity, correctness, and transparency over raw benchmark optimization, closely aligning with the goals of the hackathon.


## How the finetuning dataset is created

### Design goals

The finetuning dataset is designed to teach **explicit, judge-visible reasoning** rather than task-specific shortcuts. The primary objectives of the dataset are:

* Encourage step-by-step reasoning that is readable and self-contained
* Enforce a strict separation between reasoning and final answer
* Avoid conversational, multi-turn, or chat-style artifacts
* Ensure compatibility with deterministic, judge-equivalent evaluation

The dataset is intentionally heterogeneous in task type, but homogeneous in **output structure**.

---

### Data sources

All data is sourced from **publicly available datasets**. No proprietary or synthetic closed-source data is used.

The final mixture includes samples derived from:

* GSM8K (mathematical reasoning)
* UltraFeedback (instruction-following with reasoning)
* S1K-1.1 (reasoning trajectories)
* Tulu-CoT
* Reflection-Qwen
* LogiQA 2.0 (logical reasoning)
* MBPP (code reasoning and explanation)
* SciQ (science explanations)
* OpenOrca / OpenHermes (selectively sampled)

These datasets were chosen to cover complementary reasoning modes:
numerical, logical, explanatory, abstract, and decision-oriented reasoning.

---

### Normalization strategy

All samples are normalized into a **single, consistent structure** regardless of source.
Each example is converted into a two-part format:

* A reasoning section containing the full explanation
* A concise final answer section with no redundant reasoning

This normalization step removes all dataset-specific formatting differences and prevents the model from learning source-specific quirks.

---

### Aggressive filtering

Before normalization, each dataset undergoes multiple filtering passes to remove low-quality or incompatible samples.

Filtering criteria include:

* Removal of non-English scripts (e.g., CJK, Arabic, Devanagari)
* Removal of conversational or multi-turn exchanges
* Removal of chain-of-thought meta commentary
* Removal of truncated, partial, or derivational answers
* Removal of samples containing markdown, XML, or template artifacts
* Removal of answers that repeat reasoning content verbatim
* Rejection of overly long final answers (greater than 25 words)

For multiple-choice datasets, answers are expanded from letter labels into full natural-language options to avoid shortcut learning.

---

### Quality enforcement

Additional quality checks are applied after normalization:

* Reasoning must exceed a minimum length threshold
* Final answers must be concise and self-contained
* Reasoning and answer must not overlap semantically
* No control tokens or chat artifacts are allowed

Samples that fail any check are discarded rather than repaired.

This conservative approach favors **data quality over dataset size**.

---

### Final dataset construction

After filtering and normalization:

* All samples are merged into a single JSONL file
* The dataset is globally shuffled to avoid source clustering
* Only samples passing a final safety and consistency pass are retained

The resulting dataset is saved as:

`unified_training_data.jsonl`

---

### Public accessibility

All underlying data sources are publicly available and well-documented.
The processed dataset is derived exclusively from these sources and is made publicly accessible for evaluation.

This ensures:

* Transparency of data provenance
* Reproducibility of the approach
* Fair evaluation by judges and the community

---

### Key takeaway

The dataset is not optimized for raw scale, but for **structural consistency, reasoning clarity, and evaluation safety**. By aggressively filtering and unifying diverse public datasets into a single strict format, the model learns to reason explicitly and reliably under judge-equivalent conditions.


### Library Imports & Installations

In [ ]:
!apt-get update
!apt-get install -y git

In [ ]:

!pip install google-tunix[prod]==0.1.5

In [ ]:
# ============================================================
# 1. INSTALL + IMPORT REQUIRED LIBRARIES
# ============================================================

!pip install datasets --quiet

import json
import random
#from datasets import load_dataset
from tqdm import tqdm
import os

print("All core libraries imported successfully.")

In [ ]:
PROMPT_TEMPLATE = """Answer the following question.

You must respond in the following format:

<reasoning>model_reasoning_trace</reasoning>
<answer>model_final_answer</answer>

Question:
{question}
"""

## How the finetuning dataset is created

The finetuning dataset is constructed to explicitly teach **judge-visible reasoning**, not hidden chain-of-thought or prompt-following shortcuts.

All samples are derived exclusively from **publicly available datasets** and normalized into a **single, strict XML-style format** consisting of one reasoning block and one final answer block.

### Data sources

The final dataset is built from a mixture of reasoning-heavy sources, including:

- GSM8K (mathematical reasoning)
- UltraFeedback (instruction-following with explanations)
- S1K-1.1 (reasoning trajectories)
- Tulu-CoT
- Reflection-Qwen
- LogiQA 2.0 (logical reasoning)
- MBPP (code reasoning)
- SciQ (scientific explanations)

These datasets are selected to cover diverse reasoning modes: numerical, logical, explanatory, abstract, and judgment-based reasoning.

### Aggressive filtering and normalization

Each dataset undergoes multiple filtering passes:

- Removal of non-English scripts
- Removal of conversational or multi-turn artifacts
- Removal of chain-of-thought meta commentary
- Removal of truncated or derivational answers
- Enforcement of concise final answers (≤ 25 words)
- Rejection of samples where the answer duplicates the reasoning
- Expansion of multiple-choice answers from labels to full text

All surviving samples are normalized into the same XML structure as the prompt


Samples that fail any structural or quality constraint are discarded rather than repaired.  
This conservative approach prioritizes **data quality and evaluation safety** over dataset size.




In [ ]:
# =========================================
# Build Training Samples (FINAL, CLEAN NOT DROP)
# =========================================

import json
import re

dataset_path = "/kaggle/input/gtomsdataset/unified_training_data.jsonl"

train_samples = []

# ------------------------------------------------------------
# SAFER QUESTION EXTRACTION
# ------------------------------------------------------------
def extract_question(raw: str) -> str:
    """
    Extract the actual question text.
    Handles inputs with multiple 'Question:' blocks safely.
    """
    if not raw:
        return ""
    parts = re.split(r"\bQuestion:\b", raw)
    return parts[-1].strip()


# ------------------------------------------------------------
# REGEX HELPERS
# ------------------------------------------------------------
RE_REASONING = re.compile(r"<reasoning>\s*(.*?)\s*</reasoning>", re.S)
RE_ANSWER = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.S)

MIN_REASONING_WORDS = 10
MAX_ANSWER_WORDS = 12


# ------------------------------------------------------------
# CLEANUP PHRASES (REMOVE, NOT DROP)
# ------------------------------------------------------------
GENERIC_REASONING_PHRASES = [
    "the correct answer can be determined",
    "the problem asks",
    "this problem",
    "we are given",
    "let's answer step by step",
]

ANSWER_PREFIX_PHRASES = [
    "the answer:",
    "answer:",
    "ans:",
]

NLI_PATTERNS = [
    "premise",
    "hypothesis",
    "entailed",
    "entailment",
    "do we know",
]


def clean_text(text: str, phrases):
    """Remove bad phrases but keep remaining content."""
    lowered = text.lower()
    for p in phrases:
        if p in lowered:
            text = re.sub(re.escape(p), "", text, flags=re.IGNORECASE)
            lowered = text.lower()
    return text.strip()


# ------------------------------------------------------------
# ✅ NEW — OPTION EXTRACTION (A/B/C/D → FULL TEXT)
# ------------------------------------------------------------
OPTION_RE = re.compile(r"\b([A-E])\.\s*(.+)")

def extract_options(question: str):
    """
    Extract MCQ options from question text.
    Returns dict like {"A": "option text", ...}
    """
    options = {}
    for line in question.splitlines():
        m = OPTION_RE.match(line.strip())
        if m:
            options[m.group(1)] = m.group(2).strip()
    return options


# ------------------------------------------------------------
# BUILD TRAINING SAMPLES
# ------------------------------------------------------------
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:

        if not line.strip():
            continue

        item = json.loads(line)
        if not isinstance(item, dict):
            continue

        raw_input = item.get("input_text", "")
        target = (item.get("target_text") or "").strip()

        # ----------------------------------------------------
        # HARD XML STRUCTURE CHECK
        # ----------------------------------------------------
        if not target.startswith("<reasoning>"):
            continue
        if "</reasoning>" not in target:
            continue
        if "<answer>" not in target or "</answer>" not in target:
            continue
        if target.index("</reasoning>") > target.index("<answer>"):
            continue

        # No control tokens
        if any(tok in target for tok in ["<start_of_turn>", "<end_of_turn>", "<bos>", "<eos>"]):
            continue

        # No text after </answer>
        if target[target.index("</answer>") + len("</answer>"):].strip():
            continue

        # ----------------------------------------------------
        # EXTRACT XML CONTENT
        # ----------------------------------------------------
        r_match = RE_REASONING.search(target)
        a_match = RE_ANSWER.search(target)

        if not r_match or not a_match:
            continue

        reasoning = r_match.group(1).strip()
        answer = a_match.group(1).strip()

        # ----------------------------------------------------
        # CLEAN REASONING (DO NOT DROP)
        # ----------------------------------------------------
        reasoning = clean_text(reasoning, GENERIC_REASONING_PHRASES)

        if len(reasoning.split()) < MIN_REASONING_WORDS:
            continue

        # ----------------------------------------------------
        # QUESTION EXTRACTION
        # ----------------------------------------------------
        question = extract_question(raw_input)
        if not question:
            continue

        # Remove NLI / entailment questions entirely
        if any(p in question.lower() for p in NLI_PATTERNS):
            continue

        # ----------------------------------------------------
        # CLEAN + FIX ANSWER (DO NOT DROP)
        # ----------------------------------------------------
        answer = clean_text(answer, ANSWER_PREFIX_PHRASES)

        if not answer:
            continue

        # ✅ FIX: expand MCQ letter → full option text
        if re.fullmatch(r"[A-E]", answer):
            options = extract_options(question)
            if answer in options:
                answer = options[answer]
            else:
                continue  # option text not found → skip

        if len(answer.split()) > MAX_ANSWER_WORDS:
            continue

        # ----------------------------------------------------
        # REBUILD CLEAN TARGET
        # ----------------------------------------------------
        clean_target = (
            "<reasoning>\n"
            f"{reasoning}\n"
            "</reasoning>\n"
            "<answer>\n"
            f"{answer}\n"
            "</answer>"
        )

        train_samples.append({
            "question": question,
            "target_text": clean_target
        })


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------
print(f"✓ Loaded {len(train_samples)} cleaned training samples\n")

print("Sample records:")
for s in train_samples[:5]:
    print("-" * 80)
    print("QUESTION:", s["question"])
    print("TARGET:", s["target_text"])



## Base Model Source and Loading Method

The base model used in this project is **Gemma-3-1B (instruction-tuned)**, loaded from a **Kaggle-mounted safetensors checkpoint** using the official **Tunix Gemma3 Transformers loader**.

Specifically:

* **Base model**: `google/gemma-3-1b-it` (Gemma-3-1B Instruction-Tuned)
* **Source**: Pre-downloaded safetensors hosted as a Kaggle dataset
* **Storage format**: Safetensors
* **Architecture**: Gemma-3-1B
* **Framework**: Tunix + Transformers
* **Execution mode**: Safe single–tensor-parallel mode (`tp = 1`)

The pretrained weights are accessed directly from the local Kaggle input directory and restored using Tunix’s `params_safetensors` interface. No Hugging Face APIs or online model downloads are used at runtime.

A device mesh is constructed in **FSDP-only mode**, with tensor parallelism explicitly disabled to ensure compatibility with the safetensors format. This allows efficient data-parallel execution while preserving the original checkpoint structure.

The conversion into a JAX/Flax NNX model is handled internally by Tunix’s official Gemma3 loader, without any external dependency on Transformers-based loading pipelines.

No weights are modified, merged, quantized, or altered during initialization. The pretrained parameters are restored exactly as provided in the safetensors files.

All learning occurs **exclusively during GRPO fine-tuning**, and the trained parameters are saved separately using **Orbax** checkpointing.

The tokenizer is loaded using Tunix’s native `create_tokenizer` interface, which provides the original Gemma-3 SentencePiece tokenizer. Since Gemma-3 does not define a padding token, the EOS token is reused as the padding ID.

This loading strategy ensures:

* Use of a locally hosted, verifiable base model
* Independence from external model hosting services at runtime
* Full reproducibility in Kaggle environments
* Exact alignment with competition and hackathon rules




In [ ]:
# ============================================
#  LOAD GEMMA3 (KAGGLE SAFETENSORS) + TOKENIZER
# ============================================

import functools
import humanize
import jax
import numpy as np
from jax.sharding import Mesh
from flax import nnx

# Tunix Gemma3
from tunix.models.gemma3 import model as model_lib
from tunix.models.gemma3 import params_safetensors as params_lib
from tunix.models.gemma3.params import create_tokenizer


# --------------------------------------------
# Kaggle-mounted safetensors path
# --------------------------------------------

LOCAL_MODEL_PATH = "/kaggle/input/gemma-3/transformers/gemma-3-1b-it/1"

print("Using Kaggle safetensors model path:")
print(LOCAL_MODEL_PATH)


# --------------------------------------------
# Optional — Memory Usage
# --------------------------------------------

def show_hbm_usage():
    fmt = functools.partial(humanize.naturalsize, binary=True)

    print("\n--- DEVICE MEMORY ---")

    for i, d in enumerate(jax.local_devices()):
        stats = d.memory_stats()

        used = stats.get("bytes_in_use", 0)
        limit = stats.get("bytes_limit", 0)

        pct = (used / limit * 100) if limit else 0

        print(f"Device {i}: {fmt(used)} / {fmt(limit)} ({pct:.2f}%)")

    print("---------------------\n")


print("\nMEMORY BEFORE LOAD:")
try:
    show_hbm_usage()
except Exception:
    pass


# --------------------------------------------
# SAFE MESH (HF COMPATIBLE — tp = 1)
# --------------------------------------------

print("\nSetting up device mesh...")

devices = jax.devices()
num_devices = len(devices)

print("Detected devices:", num_devices)


# ✅ ALWAYS use tp=1 for HF safetensors
# fsdp = num_devices, tp = 1

mesh = Mesh(
    np.array(devices).reshape((num_devices, 1)),
    axis_names=("fsdp", "tp"),
)

print("Using FSDP-only mesh (tp=1, HF compatible)")
print("Mesh:", mesh, "\n")


# --------------------------------------------
# Create Gemma3 Config
# --------------------------------------------

config = model_lib.ModelConfig.gemma3_1b()

# ✅ Disable tensor-parallel sharding
if hasattr(config, "sharding_config") and config.sharding_config is not None:
    print("⚠️ Disabling tensor parallelism in config (HF mode)")
    config.sharding_config.tp = 1


print("Model config ready\n")


# --------------------------------------------
# Load Gemma3 from safetensors → JAX nnx
# --------------------------------------------

print("Loading Gemma3 weights...")
print("⏳ This may take 20–40 seconds...")

with mesh:
    gemma3 = params_lib.create_model_from_safe_tensors(
        LOCAL_MODEL_PATH,
        config,
        mesh,
    )

print("\n✅ Gemma3 model loaded successfully.")


# Optional: show structure
try:
    nnx.display(gemma3)
except Exception:
    print("nnx.display skipped")


print("\nMEMORY AFTER LOAD:")
try:
    show_hbm_usage()
except Exception:
    pass


# --------------------------------------------
# Expose Model
# --------------------------------------------

restored_model = gemma3

print("\nrestored_model ready.")
print("Type:", type(restored_model))


# --------------------------------------------
# Load Tokenizer
# --------------------------------------------

print("\n=== Loading Gemma3 SentencePiece Tokenizer ===")

tokenizer = create_tokenizer()

# Gemma3 has no PAD → use EOS
PAD_ID = tokenizer.eos_id()

print("Tokenizer loaded successfully:")
print(" - vocab size:", tokenizer.vocab_size())
print(" - eos id:", tokenizer.eos_id())
print(" - pad id (using eos):", PAD_ID)
print(" - type:", type(tokenizer))


print("\n=== SECTION 7 COMPLETE (HF SAFE MODE) ===\n")




## Overall Training and Evaluation Strategy

The primary goal of this project is **not benchmark accuracy**, but reliable, explicit, and judge-visible reasoning that is **structurally correct and reproducible**.

The model is trained using **GRPO (Group Relative Policy Optimization)** with no critic and no inference-time post-processing. Output format correctness is treated as a first-class learning objective.

### Two-Phase, Three-Stage GRPO Curriculum (1200 Total Steps)

Training is divided into **two major phases**, each lasting **600 steps**, and implemented through **three progressive stages**.

---

## **Phase 1 — Stabilization and Alignment (Steps 0–600)**

Phase 1 focuses on establishing stable output structure and early quality control. It consists of two stages.

### **Stage 1 — Format Lock (Steps 0–300)**

The model is strongly rewarded for producing the correct XML structure. Partial progress toward correct structure is allowed, while control tokens and chat artifacts are heavily penalized.

### **Stage 2 — Quality Enforcement (Steps 300–600)**

Exactly one reasoning block and one answer block are required. A minimum reasoning length is enforced, and stricter ordering constraints are applied with soft penalties.

This phase prevents early collapse and builds reliable formatting habits.

---

## **Phase 2 — Strict Convergence (Steps 600–1200)**

Phase 2 enforces full compliance and deterministic behavior.

### **Stage 3 — Strict Mode (Steps 600–1200)**

Only fully valid outputs are accepted. Any extra text, duplicated reasoning, banned patterns, or structural violations receive strong negative rewards.

This phase drives convergence toward judge-compliant outputs.

---

This two-phase, three-stage curriculum balances early stability with late-stage rigor, ensuring reliable convergence to structurally correct reasoning.

### Reward Design Philosophy

The reward function follows a **structure-first, quality-second, correctness-last** hierarchy.

Structural correctness is non-negotiable. Reasoning clarity is prioritized, and task correctness is rewarded only after format stability is achieved. Truncation is applied only after structural validation to avoid masking errors.

### Transparency and Debugging

A custom reward debugger tracks gate-level failures, reward distributions, and sample-level outputs throughout training. This enables precise diagnosis of failure modes and verification that fixes improve behavior across phases.




In [ ]:
# ============================================================
# FINAL TRAINING - GRPO 
# ============================================================


import re
import jax
import jax.numpy as jnp
import optax
from flax import nnx
from tunix.rl.rl_cluster import RLCluster, ClusterConfig, RLTrainingConfig, Role, Mode
from tunix.rl.rollout.base_rollout import RolloutConfig
from tunix.rl.grpo.grpo_learner import GRPOLearner, GRPOConfig
from collections import defaultdict, Counter
from datetime import datetime
import random

print("\n" + "="*80)
print(" FINAL TRAINING - GRPO ONLY (FIXED WRONG CLOSING TAG)")
print("="*80 + "\n")

# ============================================================
# CONFIGURATION
# ============================================================

BATCH_SIZE = 8
MAX_STEPS = 600
MAX_PROMPT_TOKENS = 384
MAX_GENERATE_TOKENS = 96

STAGE_1_END = 200   # format lock
STAGE_2_END = 400   # quality enforcement

print("Configuration:")
print(f"  Steps: {MAX_STEPS}")
print(f"  Stage 1 (0-200): Format lock")
print(f"  Stage 2 (200-400): Quality enforcement (10+ word reasoning)")
print(f"  Stage 3 (400+): Strict enforcement")
print(f"  📊 Progress updates: Every 10 steps")
print(f"  📋 Detailed reports: Every 100 steps\n")

# ============================================================
# DEBUGGING INFRASTRUCTURE
# ============================================================

class RewardDebugger:
    def __init__(self):
        self.reset_stats()
        self.step_history = []
        self.training_start_time = None
        self.failed_samples = []
        self.passed_samples = []
        
    def reset_stats(self):
        self.gate_failures = Counter()
        self.reward_distribution = defaultdict(int)
        self.total_samples = 0
        self.stage_samples = 0
        self.failed_samples = []
        self.passed_samples = []
        
    def log_failure(self, gate_name, reward_value):
        self.gate_failures[gate_name] += 1
        self.reward_distribution[self._bucket_reward(reward_value)] += 1
        self.total_samples += 1
        self.stage_samples += 1
        
    def log_success(self, reward_value, reason=""):
        key = f"SUCCESS_{reason}" if reason else "SUCCESS"
        self.gate_failures[key] += 1
        self.reward_distribution[self._bucket_reward(reward_value)] += 1
        self.total_samples += 1
        self.stage_samples += 1
    
    def log_failed_output(self, step, gate, prompt, completion, reward):
        """Log failed output for detailed debugging"""
        self.failed_samples.append({
            "step": step,
            "gate": gate,
            "prompt": prompt,
            "completion": completion,
            "reward": reward,
        })
    
    def log_passed_output(self, step, reward, prompt, completion):
        """Log passing output for comparison"""
        self.passed_samples.append({
            "step": step,
            "reward": reward,
            "prompt": prompt,
            "completion": completion,
        })
        
    def _bucket_reward(self, reward):
        if reward < -50:
            return "SEVERE_PENALTY (<-50)"
        elif reward < 0:
            return "PENALTY (0 to -50)"
        elif reward < 15:
            return "LOW_REWARD (0-15)"
        elif reward < 20:
            return "MEDIUM_REWARD (15-20)"
        elif reward < 25:
            return "HIGH_REWARD (20-25)"
        else:
            return "MAX_REWARD (25+)"
    
    def print_report(self, step):
        print("\n" + "="*80)
        print(f" REWARD DEBUG REPORT - Step {step}")
        print("="*80 + "\n")
        
        if self.stage_samples == 0:
            print("No samples in this window.\n")
            return
        
        print("GATE FAILURE ANALYSIS (Most Common First):")
        print("-" * 80)
        
        sorted_gates = sorted(self.gate_failures.items(), key=lambda x: x[1], reverse=True)
        
        for gate_name, count in sorted_gates[:15]:
            pct = (count / self.stage_samples) * 100
            bar_length = int(pct / 2)
            bar = "█" * bar_length
            
            if gate_name.startswith("SUCCESS"):
                marker = "✓"
            elif count / self.stage_samples > 0.5:
                marker = "❌"
            elif count / self.stage_samples > 0.2:
                marker = "⚠️"
            else:
                marker = "○"
                
            print(f"{marker} {gate_name:40s} {count:5d} ({pct:5.1f}%) {bar}")
        
        print()
        print("REWARD DISTRIBUTION:")
        print("-" * 80)
        
        sorted_rewards = sorted(self.reward_distribution.items(), 
                               key=lambda x: self._reward_sort_key(x[0]))
        
        for bucket, count in sorted_rewards:
            pct = (count / self.stage_samples) * 100
            bar_length = int(pct / 2)
            bar = "█" * bar_length
            print(f"  {bucket:25s} {count:5d} ({pct:5.1f}%) {bar}")
        
        print()
        
        success_count = sum(v for k, v in self.gate_failures.items() if k.startswith("SUCCESS"))
        failure_count = self.stage_samples - success_count
        success_rate = (success_count / self.stage_samples * 100) if self.stage_samples > 0 else 0
        
        print("SUMMARY:")
        print("-" * 80)
        print(f"  Total Samples: {self.stage_samples}")
        print(f"  Successes: {success_count} ({success_rate:.1f}%)")
        print(f"  Failures: {failure_count} ({100-success_rate:.1f}%)")
        
        if sorted_gates:
            top_issue = sorted_gates[0]
            if not top_issue[0].startswith("SUCCESS"):
                print(f"  🔴 TOP ISSUE: {top_issue[0]} ({top_issue[1]} failures)")
        
        # Time estimate
        if self.training_start_time and step > 0:
            elapsed = (datetime.now() - self.training_start_time).total_seconds()
            time_per_step = elapsed / step
            remaining_steps = MAX_STEPS - step
            estimated_remaining = remaining_steps * time_per_step
            
            print(f"  ⏱️  Elapsed: {int(elapsed//60)}m {int(elapsed%60)}s | ETA: {int(estimated_remaining//60)}m {int(estimated_remaining%60)}s")
        
        print("="*80 + "\n")
        
        self.step_history.append({
            'step': step,
            'gate_failures': dict(self.gate_failures),
            'reward_distribution': dict(self.reward_distribution),
            'total_samples': self.stage_samples,
            'success_rate': success_rate
        })
        
    def _reward_sort_key(self, bucket):
        order = {
            "SEVERE_PENALTY (<-50)": 0,
            "PENALTY (0 to -50)": 1,
            "LOW_REWARD (0-15)": 2,
            "MEDIUM_REWARD (15-20)": 3,
            "HIGH_REWARD (20-25)": 4,
            "MAX_REWARD (25+)": 5,
        }
        return order.get(bucket, 99)
    
    def print_trend_analysis(self):
        if len(self.step_history) < 2:
            return
            
        print("\n" + "="*80)
        print(" TREND ANALYSIS ACROSS TRAINING")
        print("="*80 + "\n")
        
        print("Success Rate Over Time:")
        for entry in self.step_history:
            step = entry['step']
            rate = entry['success_rate']
            bar_length = int(rate / 2)
            bar = "█" * bar_length
            print(f"  Step {step:4d}: {rate:5.1f}% {bar}")
        
        print()

debugger = RewardDebugger()

# ============================================================
# MESH SETUP
# ============================================================

mesh = jax.make_mesh((1, 4), axis_names=("fsdp", "tp"))
print(f"✓ Mesh created\n")



# ============================================================
# UTILITY FUNCTIONS
# ============================================================

XML_REASONING_RE = re.compile(r"<reasoning>\s*(.*?)\s*</reasoning>", re.S)
XML_ANSWER_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.S)

def extract_xml_content(text):
    r = XML_REASONING_RE.search(text)
    a = XML_ANSWER_RE.search(text)
    if not r or not a:
        return None, None
    return r.group(1).strip(), a.group(1).strip()

def normalize_answer(text):
    if text is None:
        return None
    text = str(text).strip().lower().replace(",", "")
    try:
        return str(float(text))
    except:
        return text

def extract_gold_answer(text):
    m = re.search(r"<answer>(.*?)</answer>", text, re.S)
    return re.sub(r"<.*?>", "", m.group(1)).strip() if m else None

# ============================================================
# REWARD FUNCTION - WITH CRITICAL FIXES FOR WRONG CLOSING TAG
# ============================================================

print("Creating reward function (with wrong closing tag fix)...\n")

def simplified_reward_fn(prompts, completions, gold_answers=None, step=None, **kwargs):
    """
    Reward function with THREE CRITICAL FIXES:
    1. Hard fail if <answer> closes with </reasoning>
    2. Move truncation AFTER structure validation
    3. Strengthen order check
    """
    
    rewards = []
    
    # Determine stage
    if step is not None:
        if step < STAGE_1_END:
            stage = 1
        elif step < STAGE_2_END:
            stage = 2
        else:
            stage = 3
    else:
        stage = 3
    
    # Handle gold_answers
    if gold_answers is None:
        gold_iter = [None] * len(completions)
    else:
        gold_iter = list(gold_answers)
    
    for comp, gold in zip(completions, gold_iter):
        comp_text = comp if isinstance(comp, str) else ""
        original_comp = comp_text  # Save original for logging
        
        # ❌❌❌ DO NOT TRUNCATE YET ❌❌❌
        # Truncation will happen AFTER all structure checks
        
        # Control tokens check
        CONTROL_TOKENS = ["<start_of_turn>", "<end_of_turn>", "<bos>"]
        
        if any(tok in comp_text for tok in CONTROL_TOKENS):
            debugger.log_failure("GATE_1_CONTROL_TOKENS", -50.0)
            rewards.append(-50.0)
            continue
        
        # Banned patterns check
        BANNED_PATTERNS = [
            "Question:", "Final Answer:", "\nQuestion", "\nAnswer:",
            "```xml", "```", "Response:", "Answer:",
        ]
        
        has_banned = any(pat in comp_text for pat in BANNED_PATTERNS)
        if has_banned:
            debugger.log_failure("GATE_2_BANNED_PATTERN", -30.0)
            rewards.append(-30.0)
            continue
        
        # ✅ FIX 1: HARD FAIL if <answer> closes with </reasoning>
        if "<answer>" in comp_text and "</reasoning>" in comp_text:
            a_idx = comp_text.find("<answer>")
            r_close_idx = comp_text.find("</reasoning>", a_idx)
            a_close_idx = comp_text.find("</answer>", a_idx)
            
            # If </reasoning> appears after <answer> and before </answer> (or instead of it)
            if r_close_idx != -1 and (a_close_idx == -1 or r_close_idx < a_close_idx):
                debugger.log_failure("GATE_ANSWER_CLOSED_AS_REASONING", -40.0)
                if len(prompts) > 0 and len(debugger.failed_samples) < 20:
                    debugger.log_failed_output(step if step else 0, "GATE_ANSWER_CLOSED_AS_REASONING",
                                              prompts[0] if isinstance(prompts, list) else str(prompts)[:200],
                                              comp_text, -40.0)
                rewards.append(-40.0)
                continue
        
        # Partial credit system
        partial_score = 0.0
        
        has_reasoning_open = "<reasoning>" in comp_text
        has_reasoning_close = "</reasoning>" in comp_text
        has_answer_open = "<answer>" in comp_text
        has_answer_close = "</answer>" in comp_text
        
        if has_reasoning_open:
            partial_score += 3.0
        if has_reasoning_close:
            partial_score += 3.0
        if has_answer_open:
            partial_score += 3.0
        if has_answer_close:
            partial_score += 3.0
        
        if comp_text.lstrip().startswith("<reasoning>"):
            partial_score += 5.0
        
        # Stage 1: Accept partial progress (order-aware)
        if stage == 1:
            starts_with_reasoning = comp_text.lstrip().startswith("<reasoning>")
            
            if partial_score >= 8.0 and starts_with_reasoning:
                debugger.log_success(partial_score + 10.0, "CORRECT_STRUCTURE")
                rewards.append(partial_score + 10.0)
                continue
            elif partial_score >= 8.0:
                debugger.log_success(partial_score + 3.0, "WRONG_ORDER")
                rewards.append(partial_score + 3.0)
                continue
            elif partial_score >= 5.0:
                debugger.log_success(partial_score + 2.0, "PARTIAL")
                rewards.append(partial_score + 2.0)
                continue
            elif partial_score > 0:
                debugger.log_success(partial_score, "MINIMAL")
                rewards.append(partial_score)
                continue
            else:
                debugger.log_failure("STAGE1_NO_STRUCTURE", 0.0)
                if len(prompts) > 0:
                    debugger.log_failed_output(step if step else 0, "STAGE1_NO_STRUCTURE", 
                                              prompts[0] if isinstance(prompts, list) else str(prompts)[:200], 
                                              comp_text, 0.0)
                rewards.append(0.0)
                continue
        
        # Whitespace-safe check
        clean_comp = comp_text.lstrip()
        if not clean_comp.startswith("<reasoning>"):
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure("GATE_3_MUST_START_WITH_REASONING", penalty)
            if len(prompts) > 0:
                debugger.log_failed_output(step if step else 0, "GATE_3_MUST_START_WITH_REASONING", 
                                          prompts[0] if isinstance(prompts, list) else str(prompts)[:200], 
                                          comp_text, penalty)
            if stage == 2 and partial_score > 0:
                rewards.append(max(partial_score * 0.5, penalty))
            else:
                rewards.append(max(partial_score, penalty))
            continue
        
        if comp_text.count("<reasoning>") != 1:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure(f"GATE_4_REASONING_COUNT", penalty)
            rewards.append(penalty)
            continue
        
        if comp_text.count("</reasoning>") != 1:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure(f"GATE_5_REASONING_CLOSE_COUNT", penalty)
            rewards.append(penalty)
            continue
        
        if comp_text.count("<answer>") != 1:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure(f"GATE_6_ANSWER_COUNT", penalty)
            rewards.append(penalty)
            continue
        
        if comp_text.count("</answer>") != 1:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure(f"GATE_7_ANSWER_CLOSE_COUNT", penalty)
            rewards.append(penalty)
            continue

        end_idx = comp_text.find("</answer>")
        if end_idx == -1:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure("GATE_8_NO_ANSWER_CLOSE_TAG", penalty)
            rewards.append(penalty)
            continue
        
        # ✅ FIX 3: STRENGTHEN ORDER CHECK
        try:
            r_idx = comp_text.index("<reasoning>")
            r_end_idx = comp_text.index("</reasoning>")
            a_idx = comp_text.index("<answer>")
            a_end_idx = comp_text.index("</answer>")
            
            # Strengthened check: <reasoning> must be first, </reasoning> before <answer>, <answer> before </answer>
            if not (r_idx == 0 and r_end_idx < a_idx and a_idx < a_end_idx):
                penalty = -2.0 if stage == 2 else -20.0
                debugger.log_failure("GATE_10_INVALID_TAG_ORDER", penalty)
                rewards.append(penalty)
                continue
        except ValueError:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure("GATE_11_MISSING_TAG", penalty)
            rewards.append(penalty)
            continue
        
        if re.search(r"<(?!/?(?:reasoning|answer)>)", comp_text):
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure("GATE_12_UNKNOWN_TAGS", penalty)
            rewards.append(penalty)
            continue
        
        # ✅ FIX 2: TRUNCATION AFTER ALL STRUCTURE CHECKS
        # Now that we know the structure is valid, we can safely truncate
        if "</answer>" in comp_text:
            comp_text = comp_text[: comp_text.index("</answer>") + len("</answer>")]
        
        reasoning, answer = extract_xml_content(comp_text)
        
        if not reasoning or not answer:
            penalty = -2.0 if stage == 2 else -20.0
            debugger.log_failure("GATE_13_EMPTY_CONTENT", penalty)
            rewards.append(penalty)
            continue
        
        # CHECK CONTENT LENGTH
        reason_words = len(reasoning.split())
        answer_words = len(answer.split())
        
        # Minimum requirements in stages 2+
        if stage >= 2:
            MIN_REASONING_WORDS = 10
            MIN_ANSWER_WORDS = 1
            
            if reason_words < MIN_REASONING_WORDS:
                penalty = -5.0 if stage == 2 else -25.0
                debugger.log_failure(f"GATE_14_REASONING_TOO_SHORT_{reason_words}w", penalty)
                if len(prompts) > 0:
                    debugger.log_failed_output(step if step else 0, f"GATE_14_REASONING_TOO_SHORT_{reason_words}w", 
                                              prompts[0] if isinstance(prompts, list) else str(prompts)[:200], 
                                              comp_text, penalty)
                if stage == 2:
                    partial_reward = reason_words * 0.5
                    rewards.append(max(partial_reward, penalty))
                else:
                    rewards.append(penalty)
                continue
            
            if answer_words < MIN_ANSWER_WORDS:
                penalty = -5.0 if stage == 2 else -25.0
                debugger.log_failure(f"GATE_15_ANSWER_TOO_SHORT_{answer_words}w", penalty)
                rewards.append(penalty)
                continue
        
        # ========================================================
        # ✅ GOOD FORMAT + GOOD CONTENT - CALCULATE REWARD
        # ========================================================
        
        r = 5.0  # Base
        
        # Reasoning quality (main reward)
        if reason_words >= 20:
            r += 15.0  # Excellent
        elif reason_words >= 15:
            r += 12.0  # Good
        elif reason_words >= 10:
            r += 8.0   # Acceptable
        else:
            r += 3.0   # Minimal
        
        # Answer length
        if 1 <= answer_words <= 20:
            r += 3.0
        elif answer_words <= 30:
            r += 2.0
        else:
            r += 1.0
        
        # Correctness (largest bonus)
        is_correct = False
        if gold is not None:
            g = normalize_answer(gold)
            a = normalize_answer(answer)
            
            if g is not None and a is not None:
                try:
                    if float(g) == float(a):
                        r += 15.0
                        is_correct = True
                except:
                    if g == a:
                        r += 15.0
                        is_correct = True
        
        r = float(max(min(r, 38.0), 5.0))
        
        if is_correct:
            debugger.log_success(r, "CORRECT_ANSWER")
        else:
            debugger.log_success(r, "WRONG_ANSWER")
        
        # Log passing sample (limit volume)
        if step is not None and len(debugger.passed_samples) < 20:
            debugger.log_passed_output(
                step,
                r,
                prompts[0] if isinstance(prompts, list) else str(prompts)[:200],
                comp_text
            )
        
        rewards.append(r)
    
    return jnp.array(rewards, dtype=jnp.float32)

# ============================================================
# WRAP REWARD FUNCTION WITH LIVE PROGRESS TRACKING
# ============================================================

_last_report_step = 0
_step_counter = 0

def tracked_reward_fn(prompts, completions, gold_answers=None, step=None, **kwargs):
    global _last_report_step, _step_counter
    
    if step is not None:
        _step_counter = step
    else:
        _step_counter += 1
        step = _step_counter
    
    if debugger.training_start_time is None:
        debugger.training_start_time = datetime.now()
    
    result = simplified_reward_fn(prompts, completions, gold_answers, step, **kwargs)
    
    # Print mini-update every 10 steps
    if step % 10 == 0:
        success_count = sum(v for k, v in debugger.gate_failures.items() if k.startswith("SUCCESS"))
        total = debugger.stage_samples
        success_rate = (success_count / total * 100) if total > 0 else 0
        
        # Determine stage
        if step < STAGE_1_END:
            stage_name = "Stage 1 (Format Lock)"
        elif step < STAGE_2_END:
            stage_name = "Stage 2 (Quality)"
        else:
            stage_name = "Stage 3 (Strict)"
        
        # Progress bar
        progress = (step / MAX_STEPS) * 100
        bar_length = 30
        filled = int((progress / 100) * bar_length)
        bar = "█" * filled + "░" * (bar_length - filled)
        
        print(f"[{bar}] {progress:5.1f}% | Step {step:4d}/{MAX_STEPS} | {stage_name:23s} | Success: {success_rate:5.1f}% ({success_count}/{total})")
        
        # Show random failed samples
        if debugger.failed_samples:
            num_samples = min(2, len(debugger.failed_samples))
            samples_to_show = random.sample(debugger.failed_samples, num_samples)
            print(f"\n🔍 SAMPLE FAILED OUTPUTS ({num_samples} random from last 10 steps):")
            for i, sample in enumerate(samples_to_show, 1):
                print("-" * 80)
                print(f"Sample {i}:")
                print(f"  Gate   : {sample['gate']}")
                print(f"  Reward : {sample['reward']:.1f}")
                print(f"\n  PROMPT (first 200 chars):")
                print(f"  {sample['prompt'][:200]}...")
                print(f"\n  COMPLETION (full):")
                print(f"  {sample['completion']}")
                print("-" * 80)
            print()
        
        # Show random passing samples
        if debugger.passed_samples:
            num_samples = min(2, len(debugger.passed_samples))
            samples_to_show = random.sample(debugger.passed_samples, num_samples)
            print(f"\n✅ SAMPLE PASSING OUTPUTS ({num_samples} random):")
            for i, sample in enumerate(samples_to_show, 1):
                print("-" * 80)
                print(f"Sample {i}:")
                print(f"  Reward : {sample['reward']:.1f}")
                print(f"\n  PROMPT (first 200 chars):")
                print(f"  {sample['prompt'][:200]}...")
                print(f"\n  COMPLETION:")
                print(f"  {sample['completion']}")
                print("-" * 80)
            print()
    
    # Full report every 100 steps
    if step % 100 == 0 and step > 0:
        debugger.print_report(step)
        debugger.reset_stats()
        _last_report_step = step
    
    return result

# ============================================================
# TEST
# ============================================================

print("Testing reward function...\n")

tests = [
    ("Good", "<reasoning>This requires calculating 25% of 200, which equals 50</reasoning><answer>50</answer>", "50"),
    ("Wrong close", "<reasoning>Test</reasoning><answer>42</reasoning>", "42"),  # Should fail hard
    ("Missing close", "<reasoning>Test<answer>42</answer>", "42"),  # Should fail  
]

print("Testing fixes:")
for name, output, gold in tests:
    r = simplified_reward_fn(["x"], [output], [gold], step=250)
    print(f"  {name:15s} → {float(r[0]):6.1f}")

print("\n✅ Tests complete\n")
debugger.reset_stats()

# ============================================================
# REST OF THE CODE IS IDENTICAL
# ============================================================

print("="*80)
print(" NOTES:")
print("="*80)
print("✅ FIX 1: Hard -40.0 penalty if <answer> closes with </reasoning>")
print("✅ FIX 2: Truncation moved AFTER all structure checks")
print("✅ FIX 3: Strengthened order check (r_idx == 0)")
print("\nExpected: Within 20-40 steps, wrong closing tag → near zero")
print("="*80 + "\n")

# [REST OF YOUR CODE FOR DATA PREP, MODELS, GRPO SETUP, TRAINING PHASES]
# (Copy from your original file - everything after the reward function stays the same)

# ============================================================
# DATA PREPARATION
# ============================================================

print("="*80)
print(" DATA PREPARATION")
print("="*80 + "\n")

def prepare_training_data(train_samples, tokenizer):
    print(f"Processing {len(train_samples)} training samples...")
    
    def truncate_prompt(prompt):
        ids = tokenizer.encode(prompt)
        if len(ids) <= MAX_PROMPT_TOKENS:
            return prompt
        return tokenizer.decode(ids[:MAX_PROMPT_TOKENS])
    
    processed = []
    
    for sample in train_samples:
        target_text = sample.get("target_text", "").strip()
        
        for token in ["<start_of_turn>", "<end_of_turn>", "<eos>", "<bos>"]:
            target_text = target_text.replace(token, "")
        
        m = re.fullmatch(
            r"\s*<reasoning>.*?</reasoning>\s*<answer>.*?</answer>\s*",
            target_text,
            re.S
        )
        
        if not m:
            continue
        
        clean_target = m.group(0).strip()
        
        if not clean_target.startswith("<reasoning>"):
            continue
        if not clean_target.endswith("</answer>"):
            continue
        
        question = sample.get("question", "")
        if not question:
            continue
        
        # Filter boilerplate reasoning
        if "the correct answer can be determined" in clean_target.lower():
            continue
        
        # Enforce min reasoning length
        reasoning, _ = extract_xml_content(clean_target)
        if reasoning is None or len(reasoning.split()) < 10:
            continue
        
        prompt = truncate_prompt(PROMPT_TEMPLATE.format(question=question))
        gold_answer = extract_gold_answer(clean_target)
        
        processed.append({
            "question": question,
            "prompt": prompt,
            "target_text": clean_target,
            "gold_answer": gold_answer,
        })
    
    print(f"✓ Processed {len(processed)} valid samples (filtered for quality)\n")
    return processed

processed_samples = prepare_training_data(train_samples, tokenizer)

# ============================================================
# CREATE MODELS
# ============================================================

print("="*80)
print(" MODEL SETUP")
print("="*80 + "\n")

print("Creating models...")
actor_model = nnx.clone(restored_model)
reference_model = nnx.clone(restored_model)

for _, v in nnx.state(reference_model).items():
    if isinstance(v, nnx.Param):
        v.requires_grad = False

ref_params = nnx.state(reference_model, nnx.Param)
ref_params = jax.tree.map(jax.lax.stop_gradient, ref_params)
nnx.update(reference_model, ref_params)
print("✓ Models created\n")

# ============================================================
# GRPO SETUP
# ============================================================

print("="*80)
print(" GRPO SETUP")
print("="*80 + "\n")

grpo_optimizer = optax.adamw(learning_rate=3e-6, b1=0.9, b2=0.99, weight_decay=0.0)

training_config = RLTrainingConfig(
    actor_optimizer=grpo_optimizer,
    critic_optimizer=None,
    mini_batch_size=1,
    train_micro_batch_size=1,
    rollout_micro_batch_size=1,
    compute_logps_micro_batch_size=1,
    max_steps=MAX_STEPS,
    eval_every_n_steps=50,
    metrics_logging_options=None,
)

print("🔧 CRITICAL: eos_tokens=[tokenizer.eos_id()] only\n")

# Phase 1 rollout config
train_rollout_cfg_phase1 = RolloutConfig(
    max_prompt_length=MAX_PROMPT_TOKENS,
    max_tokens_to_generate=MAX_GENERATE_TOKENS,
    kv_cache_size=4096,
    temperature=1.0,  # Higher temp for exploration
    top_k=50,
    top_p=0.95,
    eos_tokens=[
    tokenizer.eos_id(),
    tokenizer.piece_to_id("<end_of_turn>"),
],

)

# 🔧 FIX 4: Phase 2 rollout config with REDUCED TEMPERATURE
train_rollout_cfg_phase2 = RolloutConfig(
    max_prompt_length=MAX_PROMPT_TOKENS,
    max_tokens_to_generate=MAX_GENERATE_TOKENS,
    kv_cache_size=4096,
    temperature=0.6,  # Reduced from 1.0 - less chatter
    top_k=50,
    top_p=0.95,
    eos_tokens=[
    tokenizer.eos_id(),
    tokenizer.piece_to_id("<end_of_turn>"),
],

)

eval_rollout_cfg = RolloutConfig(
    max_prompt_length=MAX_PROMPT_TOKENS,
    max_tokens_to_generate=MAX_GENERATE_TOKENS,
    kv_cache_size=4096,
    temperature=0.0,
    top_k=1,
    top_p=None,
    eos_tokens=[
    tokenizer.eos_id(),
    tokenizer.piece_to_id("<end_of_turn>"),
],

)

print(f"✓ EOS tokens: [{tokenizer.eos_id()}]\n")

# Phase 1 cluster
cluster_config_phase1 = ClusterConfig(
    role_to_mesh={
        Role.ACTOR: mesh,
        Role.REFERENCE: mesh,
        Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    training_config=training_config,
    rollout_config={
        Mode.TRAIN: train_rollout_cfg_phase1,
        Mode.EVAL: eval_rollout_cfg,
    },
)

print("Building Phase 1 RL cluster...")
rl_cluster_phase1 = RLCluster(
    actor=actor_model,
    reference=reference_model,
    critic=None,
    reward=None,
    tokenizer=tokenizer,
    cluster_config=cluster_config_phase1,
)
print("✓ Phase 1 RL cluster built\n")

print("Preparing batches...")

train_ds = []
num_devices = jax.device_count()
batch_unit = BATCH_SIZE * num_devices
n = len(processed_samples)

for i in range(0, n, batch_unit):
    if len(train_ds) >= MAX_STEPS:
        break
    
    idxs = list(range(i, min(i + batch_unit, n)))
    while len(idxs) < batch_unit:
        idxs.append(idxs[-1])
    
    for j in range(0, batch_unit, BATCH_SIZE):
        batch_idxs = idxs[j:j + BATCH_SIZE]
        train_ds.append({
            "prompts": [processed_samples[x]["prompt"] for x in batch_idxs],
            "gold_answers": [processed_samples[x]["gold_answer"] for x in batch_idxs],
        })

print(f"✓ {len(train_ds)} batches ready\n")

# ============================================================
# PHASE 1: FORMAT LOCK (Steps 0-200)
# ============================================================

print("="*80)
print(" PHASE 1: FORMAT LOCK (Steps 0-200)")
print("="*80 + "\n")

print("🔒 PHASE 1 CONFIG:")
grpo_cfg_phase1 = GRPOConfig(
    num_generations=4,   # Exploration
    num_iterations=1,
    beta=0.0,            # No KL pressure
    epsilon=0.2,
)
print("   • num_generations=4")
print("   • beta=0.0 (no KL pressure)")
print("   • temperature=1.0 (exploration)")
print("   • Goal: Lock XML format via reward gates\n")

print("Creating Phase 1 learner...")
learner_phase1 = GRPOLearner(
    rl_cluster=rl_cluster_phase1,
    reward_fns=tracked_reward_fn,
    algo_config=grpo_cfg_phase1,
)
print("✓ Phase 1 learner created\n")

print("✅ LIVE PROGRESS ENABLED:")
print("   • Progress bar every 10 steps")
print("   • Failed & passing samples displayed")
print("   • Detailed report every 100 steps\n")

print("Expected: Model learns <reasoning>...</reasoning><answer>...</answer> format\n")

print("Starting Phase 1 training...\n")

phase1_steps = min(STAGE_1_END, len(train_ds))

try:
    learner_phase1.train(train_ds[:phase1_steps])
except KeyboardInterrupt:
    print("\n\n" + "="*80)
    print(" PHASE 1 INTERRUPTED BY USER")
    print("="*80)
    print("\nGenerating Phase 1 final report...\n")
    if debugger.stage_samples > 0:
        debugger.print_report(_step_counter)

print("\n" + "="*80)
print(" PHASE 1 COMPLETE - FORMAT LOCKED")
print("="*80 + "\n")

# ============================================================
# SYNC REFERENCE MODEL (CRITICAL FOR PHASE 2 SUCCESS)
# ============================================================

print("="*80)
print(" 🔁 SYNCING REFERENCE MODEL AFTER PHASE 1")
print("="*80 + "\n")

print("Updating reference model to match actor...")
actor_state = nnx.state(actor_model, nnx.Param)
ref_state = jax.tree.map(jax.lax.stop_gradient, actor_state)
nnx.update(reference_model, ref_state)

print("✅ Reference model updated — KL will preserve XML format\n")

# ============================================================
# PHASE 2: QUALITY REFINEMENT (Steps 200-600)
# ============================================================

print("="*80)
print(" PHASE 2: QUALITY REFINEMENT (Steps 200-600)")
print("="*80 + "\n")

# 🔧 FIX 4: REDUCED HYPERPARAMETERS
print("🚀 PHASE 2 CONFIG (REDUCED VERBOSITY):")
grpo_cfg_phase2 = GRPOConfig(
    num_generations=6,   # Reduced from 8
    num_iterations=1,
    beta=0.004,          # Softer KL (was 0.006)
    epsilon=0.2,
)
print("   • num_generations=6 (focused, not 8)")
print("   • beta=0.004 (softer KL)")
print("   • temperature=0.6 (reduced chatter)")
print("   • Goal: Refine reasoning quality & correctness\n")

# Create Phase 2 cluster with reduced temperature
cluster_config_phase2 = ClusterConfig(
    role_to_mesh={
        Role.ACTOR: mesh,
        Role.REFERENCE: mesh,
        Role.ROLLOUT: mesh,
    },
    rollout_engine="vanilla",
    training_config=training_config,
    rollout_config={
        Mode.TRAIN: train_rollout_cfg_phase2,  # Lower temperature
        Mode.EVAL: eval_rollout_cfg,
    },
)

print("Building Phase 2 RL cluster...")
rl_cluster_phase2 = RLCluster(
    actor=actor_model,  # SAME model (continues from Phase 1)
    reference=reference_model,  # Updated reference
    critic=None,
    reward=None,
    tokenizer=tokenizer,
    cluster_config=cluster_config_phase2,
)
print("✓ Phase 2 RL cluster built\n")

print("Creating Phase 2 learner...")
learner_phase2 = GRPOLearner(
    rl_cluster=rl_cluster_phase2,
    reward_fns=tracked_reward_fn,
    algo_config=grpo_cfg_phase2,
)
print("✓ Phase 2 learner created\n")

print("Expected: Within 20-40 steps → 40-60% success → gradual climb to 70-85%\n")

print("Starting Phase 2 training...\n")

try:
    learner_phase2.train(train_ds[phase1_steps:])
except KeyboardInterrupt:
    print("\n\n" + "="*80)
    print(" PHASE 2 INTERRUPTED BY USER")
    print("="*80)
    print("\nGenerating Phase 2 final report...\n")

print("\n" + "="*80)
print(" TRAINING COMPLETE")
print("="*80)

# Print final report if we have data
if debugger.stage_samples > 0:
    debugger.print_report(_step_counter)

debugger.print_trend_analysis()

print("\n✅ ALL FIXES APPLIED:")
print("   • Logical truncation after </answer>")
print("   • Soft penalties for text-after-answer")
print("   • Whitespace-safe checks")
print("   • Reduced Phase 2 hyperparameters")
print("   • Passing & failing sample visibility")
print("\n💡 TIP: You can press Ctrl+C to stop training early and still get a report!")

## Model saving format (Flax / Orbax)

The trained model is saved using **Flax NNX parameters serialized with Orbax**, which is the native checkpointing format used by **Tunix and JAX**.

### Saving methodology

After GRPO training completes, the actor model’s parameters are extracted directly from the Flax NNX model:

- Parameters are obtained via `nnx.state(actor_model, nnx.Param)`
- The resulting object is a JAX PyTree containing Flax parameter tensors
- No conversion to PyTorch or Hugging Face formats is performed

The parameters are then saved using `orbax.checkpoint.StandardCheckpointer`, producing a structured checkpoint directory containing:

- Parameter shards
- Shape and dtype metadata
- Checkpoint manifests required for safe restoration

### Key properties

- **Framework**: Flax NNX + JAX
- **Checkpoint system**: Orbax
- **Format**: Native Flax parameter PyTree (not PyTorch, not safetensors)
- **Device-safe**: Compatible with TPU and multi-device meshes
- **Deterministic restore**: Requires an abstract parameter tree for shape validation

Each checkpoint is saved under a timestamped directory to ensure consistency and avoid reliance on trainer step counters.

### Restoration compatibility

The saved checkpoints are restored by:

1. Instantiating the base Gemma-3-1B model architecture
2. Creating an abstract parameter tree matching the model structure
3. Restoring Orbax parameters into the model via `nnx.update`

This guarantees that evaluation and inference use **exactly the trained Flax weights**, with no post-processing, merging, or hidden transformations.

### Public availability

The Flax / Orbax checkpoint directory is uploaded to Kaggle and made publicly accessible, allowing judges to load the model using the same Tunix modeling code used during training.


In [ ]:
# ============================================
# SAVE TRAINED MODEL (TUNIX / ORBAX)
# ============================================

import os
import shutil
import orbax.checkpoint as ocp
from flax import nnx
import jax
from datetime import datetime

print("\n" + "=" * 80)
print(" SAVING TRAINED ACTOR MODEL (ORBAX / TUNIX FORMAT)")
print("=" * 80 + "\n")

# ============================================================
# PATHS
# ============================================================

CKPT_DIR = "/kaggle/working/ckpt"
ACTOR_DIR = os.path.join(CKPT_DIR, "actor")

# Use timestamp instead of trainer step (safer & always defined)
STEP_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
FINAL_CKPT_PATH = os.path.join(ACTOR_DIR, STEP_TAG)

# ============================================================
# CLEAN OLD CHECKPOINTS
# ============================================================

if os.path.exists(CKPT_DIR):
    print("⚠️ Removing old checkpoint directory")
    shutil.rmtree(CKPT_DIR)

os.makedirs(ACTOR_DIR, exist_ok=True)

# ============================================================
# EXTRACT TRAINED PARAMETERS (THIS IS YOUR MODEL)
# ============================================================

print("📦 Extracting trained actor parameters...")

# actor_model is already TRAINED by GRPO
params = nnx.state(actor_model, nnx.Param)

print(f"✓ Extracted {len(params)} parameter tensors")

# ============================================================
# SAVE USING ORBAX
# ============================================================

checkpointer = ocp.StandardCheckpointer()

print(f"💾 Saving checkpoint to:\n{FINAL_CKPT_PATH}")

checkpointer.save(FINAL_CKPT_PATH, params)

# 🚨 REQUIRED — BLOCK UNTIL SAVE IS COMPLETE
checkpointer.wait_until_finished()

print("✅ Orbax save finalized")

# ============================================================
# HARD SANITY CHECKS (IMPORTANT)
# ============================================================

assert os.path.isdir(FINAL_CKPT_PATH), "❌ Checkpoint directory missing"

metadata_path = os.path.join(FINAL_CKPT_PATH, "_CHECKPOINT_METADATA")
assert os.path.exists(metadata_path), "❌ Missing _CHECKPOINT_METADATA"

print("✅ Checkpoint integrity verified")

print("\n📂 Checkpoint contents:")
for f in os.listdir(FINAL_CKPT_PATH):
    print("  •", f)

print("\n" + "=" * 80)
print(" MODEL SAVE COMPLETE")
print("=" * 80)


## Unrestricted (Multi-Session) Model Submission

This project also participates in the **optional Unrestricted (Multi-Session) evaluation mode**.

### Kaggle model ID

omshreegyanraj/gemma3-1b-grpov1f1

### Model provenance

- **Base model**: `google/gemma-3-1b-it`
- **Architecture**: Gemma-3-1B
- **Framework**: Tunix + JAX + Flax NNX
- **Post-training method**: GRPO (Group Relative Policy Optimization)

The unrestricted model is built from the **same base Gemma-3-1B architecture** used in the single-session submission, fully complying with hackathon requirements.


In [ ]:

unrestricted_kaggle_model = "omshreegyanraj/gemma3-1b-grpov1f1"  

In [ ]:
# ============================================================
# DEBUG: INSPECT UNRESTRICTED MODEL STRUCTURE (READ-ONLY)
# ============================================================

import os
import kagglehub

UNRESTRICTED_MODEL = "omshreegyanraj/gemma3-1b-grpov1f1"

print("=" * 80)
print("⬇️ DOWNLOADING UNRESTRICTED MODEL (FOR DEBUG ONLY)")
print("=" * 80)

version_root = kagglehub.model_download(f"{UNRESTRICTED_MODEL}/jax/default")

print("\n📂 VERSION ROOT:")
print(version_root)
print("\n📂 TREE (depth-limited):\n")

def print_tree(root, prefix="", max_depth=6, depth=0):
    if depth > max_depth:
        print(prefix + "└── ...")
        return
    try:
        entries = sorted(os.listdir(root))
    except Exception as e:
        print(prefix + f"[ERROR reading dir: {e}]")
        return

    for i, name in enumerate(entries):
        path = os.path.join(root, name)
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + name)
        if os.path.isdir(path):
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(path, new_prefix, max_depth, depth + 1)

print_tree(version_root, max_depth=8)




In [ ]:
# ============================================================
# JUDGE-COMPATIBLE EVALUATION + INTERNAL FORMAT VALIDATION
# ============================================================

import os
import re
import jax
from flax import nnx
import orbax.checkpoint as ocp
import kagglehub

from tunix.models.gemma3 import model as model_lib
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib


# ============================================================
# FORMAT TOKENS
# ============================================================

REASONING_START = "<reasoning>"
REASONING_END   = "</reasoning>"
SOLUTION_START  = "<answer>"
SOLUTION_END    = "</answer>"


# ============================================================
# CONFIG
# ============================================================

MAX_PROMPT_LENGTH = 1024
MAX_GENERATION_STEPS = 512

INF_TEMPERATURE = 0.0
INF_TOP_K = 1
INF_TOP_P = None
SEED = 0


# ============================================================
# LOAD TOKENIZER
# ============================================================

BASE_GEMMA = "google/gemma-3/flax/gemma3-1b-it"

base_root = kagglehub.model_download(BASE_GEMMA)

tokenizer_path = os.path.join(base_root, "tokenizer.model")

tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=tokenizer_path)

print("✓ Tokenizer loaded")


# ============================================================
# LOAD MODEL
# ============================================================

MODEL_ID = "omshreegyanraj/gemma3-1b-grpov1f1"

version_root = kagglehub.model_download(f"{MODEL_ID}/jax/default")


model_config = model_lib.ModelConfig.gemma3_1b()

model = model_lib.Gemma3(
    config=model_config,
    rngs=nnx.Rngs(SEED),
)

print("✓ Model created")


# ============================================================
# LOAD CHECKPOINT
# ============================================================

actor_dir = os.path.join(version_root, "actor")

latest_step = max(
    int(d) for d in os.listdir(actor_dir)
    if d.isdigit()
)

final_ckpt = os.path.join(
    actor_dir,
    str(latest_step),
    "model_params",
)


abs_params = jax.tree.map(
    lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
    nnx.state(model, nnx.Param),
)

checkpointer = ocp.StandardCheckpointer()

params = checkpointer.restore(final_ckpt, target=abs_params)


nnx.update(
    model,
    jax.tree.map(
        lambda _, new: new,
        nnx.state(model, nnx.Param),
        params,
    ),
)

print(f"✓ Restored checkpoint {latest_step}")


# ============================================================
# CREATE SAMPLER
# ============================================================

sampler = sampler_lib.Sampler(
    transformer=model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=MAX_PROMPT_LENGTH + MAX_GENERATION_STEPS + 256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

print("✓ Sampler initialized")





# ============================================================
# JUDGE CLASS (WITH BUILT-IN FORMAT CHECKER)
# ============================================================

class TunixHackathonJudge:

    def __init__(
        self,
        temperature,
        top_k,
        top_p,
        max_generation_steps,
        seed,
    ):
        self.temperature = temperature
        self.top_k = top_k
        self.top_p = top_p
        self.max_generation_steps = max_generation_steps
        self.seed = seed


    # -------------------------------
    # Internal format validator
    # -------------------------------
    def _validate_format(self, text: str):

        if not text.startswith(REASONING_START):
            return False, "Does not start with <reasoning>"

        if REASONING_END not in text:
            return False, "Missing </reasoning>"

        if SOLUTION_START not in text:
            return False, "Missing <answer>"

        if SOLUTION_END not in text:
            return False, "Missing </answer>"


        r_start = text.find(REASONING_START)
        r_end   = text.find(REASONING_END)
        a_start = text.find(SOLUTION_START)
        a_end   = text.find(SOLUTION_END)


        if not (r_start < r_end < a_start < a_end):
            return False, "Wrong tag ordering"


        # No trailing text
        tail = text[a_end + len(SOLUTION_END):].strip()

        if tail:
            return False, "Extra text after </answer>"


        # Only one block
        if text.count(REASONING_START) != 1:
            return False, "Multiple <reasoning> blocks"

        if text.count(SOLUTION_START) != 1:
            return False, "Multiple <answer> blocks"


        # Non-empty blocks
        r = re.search(r"<reasoning>(.*?)</reasoning>", text, re.S)
        a = re.search(r"<answer>(.*?)</answer>", text, re.S)

        if not r or not r.group(1).strip():
            return False, "Empty reasoning"

        if not a or not a.group(1).strip():
            return False, "Empty answer"


        return True, "OK"


    # -------------------------------
    # Evaluation
    # -------------------------------
    def evaluate(self, sampler, prompt_template, questions):

        results = []

        for i, q in enumerate(questions, 1):

            prompt = prompt_template.format(question=q)

            outputs = sampler(
                input_strings=[prompt],
                max_generation_steps=self.max_generation_steps,
                temperature=self.temperature,
                top_k=self.top_k,
                top_p=self.top_p,
            )

            text = outputs.text[0].strip()


            # Remove leaked <end_of_turn>
            if "<end_of_turn>" in text:
                text = text.split("<end_of_turn>")[0].strip()


            valid, msg = self._validate_format(text)


            results.append({
                "id": i,
                "question": q,
                "output": text,
                "valid_format": valid,
                "validation_msg": msg,
            })


        return results


# ============================================================
# RUN EVALUATION
# ============================================================

QUESTIONS = [
    "Explain why having a backup plan is important when making decisions.",
    "Why might two people interpret the same event differently?",
    "Why might increasing speed reduce accuracy in some tasks?",
    "Why does adding more people to a project sometimes slow it down?",
    "Why might a simple solution be better than a complex one?",
    "Why do rules sometimes fail in real-world situations?",
    "If something is easy to explain but hard to do, why might that be?",
    "Why can rules with many exceptions become ineffective?",
    "Explain why learning from mistakes is valuable.",
    "Explain why planning ahead can reduce stress.",
    "Why is it sometimes better to delay a decision?",
    "Should consistency matter more than flexibility in rules? Explain why.",
    "If a train travels 60 km in 1 hour, how far does it travel in 30 minutes?",
    "Explain why setting priorities is important when managing time.",
]



judge = TunixHackathonJudge(
    INF_TEMPERATURE,
    INF_TOP_K,
    INF_TOP_P,
    MAX_GENERATION_STEPS,
    SEED,
)


results = judge.evaluate(
    sampler,
    PROMPT_TEMPLATE,
    QUESTIONS,
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

for r in results:

    print("\n" + "="*90)

    print(f"QUESTION {r['id']}:")
    print(r["question"])

    print("\nMODEL OUTPUT:")
    print(r["output"])

    print("\nFORMAT CHECK:")
    print("✅ VALID" if r["valid_format"] else "❌ INVALID")
    print("Reason:", r["validation_msg"])

    print("="*90)


### Conclusion

The evaluation results indicate that the model is generally successful in producing outputs that conform to the required structural format, with the majority of responses passing the automated format checks. In most cases, the `<reasoning>` and `<answer>` blocks are correctly generated, demonstrating that the staged GRPO training strategy is effective in reinforcing structural compliance.

However, despite completing the full 1200-step training schedule, occasional format inconsistencies and structural weaknesses remain. These issues become more apparent in certain contexts, particularly in coding-related tasks and mathematical reasoning problems, where the model is more likely to deviate from the prescribed output schema. Such deviations may include incomplete tags, malformed blocks, unintended tokens, or inconsistent answer formatting.

These observations suggest that while the two-phase, three-stage curriculum significantly improves overall format stability, strict compliance is not yet fully guaranteed under all task conditions. Increased cognitive load during problem-solving—especially in multi-step coding or numerical reasoning—can interfere with the model’s ability to consistently maintain the required structure.

Furthermore, minor content-level inconsistencies, such as overly brief answers or residual artifacts, indicate that structural learning and task reasoning are still partially competing objectives during generation.

In summary, the training approach achieves strong baseline reliability and reproducibility, but occasional format breakdowns persist, particularly in technically demanding scenarios. Future work should focus on strengthening structural constraints under high-complexity tasks, improving robustness in coding and mathematical domains, and introducing additional reinforcement mechanisms to further reduce schema violations.



### Optional Code Block (Not Executable Due to Library Compatibility Issues)

The code below is provided for reference purposes only. It follows the same logic that was used to build the final public dataset and can be reviewed for strategic and methodological understanding.

However, due to library version and compatibility issues, this code is **not executable** in the current environment. Attempting to run it may result in errors.

### Data Sourcing and Loading

The following section demonstrates the original approach used for data sourcing and loading.



In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================

!pip install -q huggingface_hub pyarrow fastparquet datasets kaggle

from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download
import json, random, os
import pandas as pd
from pprint import pprint


# ============================================================
# 1. LOAD BASE DATASETS
# ============================================================

print("Loading base datasets...\n")

gsm8k = load_dataset("openai/gsm8k", "main")
orca = load_dataset("Open-Orca/OpenOrca")
sciq = load_dataset("allenai/sciq")
codealpaca = load_dataset("yahma/alpaca-cleaned")
openhermes = load_dataset("teknium/OpenHermes-2.5")

print("Base datasets loaded.\n")


# ============================================================
# 2. HELPERS
# ============================================================

def load_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            items.append(json.loads(line))
    return items


def sample(dataset, n=400, seed=42):
    dataset = dataset.shuffle(seed=seed)
    return dataset.select(range(min(n, len(dataset))))


# ============================================================
# 3. ULTRAFEEDBACK
# ============================================================

print("\nDownloading UltraFeedback...\n")

    "evol_instruct.jsonl",
    "false_qa.jsonl",
    "flan.jsonl",
    "sharegpt.jsonl",
    "truthful_qa.jsonl",
    "ultrachat.jsonl"
]

downloaded_ultra = {}
for fname in ultra_files:
    path = hf_hub_download(
        repo_id="openbmb/UltraFeedback",
        repo_type="dataset",
        filename=fname
    )
    downloaded_ultra[fname] = path
    print("Downloaded UltraFeedback:", fname)

evol       = random.sample(load_jsonl(downloaded_ultra["evol_instruct.jsonl"]), 400)
flan       = random.sample(load_jsonl(downloaded_ultra["flan.jsonl"]), 400)
sharegpt   = random.sample(load_jsonl(downloaded_ultra["sharegpt.jsonl"]), 400)
truthful   = random.sample(load_jsonl(downloaded_ultra["truthful_qa.jsonl"]), 200)
false_qa   = random.sample(load_jsonl(downloaded_ultra["false_qa.jsonl"]), 200)
ultrachat  = random.sample(load_jsonl(downloaded_ultra["ultrachat.jsonl"]), 400)

ultra_subset = evol + flan + sharegpt + truthful + false_qa + ultrachat


# ============================================================
# 4. S1K (PARQUET)
# ============================================================

s1k_path = hf_hub_download(
    repo_id="yentinglin/s1K-1.1-trl-format",
    repo_type="dataset",
    filename="data/train-00000-of-00001.parquet"
)

df_s1k = pd.read_parquet(s1k_path)
s1k_list = df_s1k.to_dict(orient="records")


# ============================================================
# 5. TULU-CoT
# ============================================================

tulu_path = hf_hub_download(
    repo_id="arazd/tulu_cot",
    repo_type="dataset",
    filename="cot_data.jsonl"
)

tulu_cot = load_jsonl(tulu_path)


# ============================================================
# 6. REFLECTION-QWEN
# ============================================================

reflection_path = hf_hub_download(
    repo_id="lemon-mint/reflection-qwen-2.5-mixture-v0.1",
    repo_type="dataset",
    filename="data/train-00000-of-00001.parquet"
)

df_reflection = pd.read_parquet(reflection_path)
reflection_list = df_reflection.to_dict(orient="records")


# ============================================================
# 7. MBPP (KAGGLE JSONL — SINGLE FILE)
# ============================================================

print("\nDownloading MBPP dataset from Kaggle...\n")

!kaggle datasets download -d mpwolke/mbppjsonl -p ./mbpp --unzip

mbpp_all = load_jsonl("./mbpp/mbpp.jsonl")
random.shuffle(mbpp_all)

mbpp_val   = mbpp_all[:200]
mbpp_train = mbpp_all[200:]
mbpp_test  = []

print(f"MBPP total: {len(mbpp_all)}")
print(f"MBPP train: {len(mbpp_train)}")
print(f"MBPP val:   {len(mbpp_val)}")


# ============================================================
# 8. LOGIQA 2.0 (GITHUB)
# ============================================================

print("\nLoading LogiQA2.0 from GitHub...\n")

base_url = "https://raw.githubusercontent.com/csitfun/LogiQA2.0/main/logiqa/DATA/LOGIQA/"
paths = {
    "train": base_url + "train.txt",
    "val":   base_url + "dev.txt",
    "test":  base_url + "test.txt",
}

def download_logiqa(url):
    !wget -q {url} -O tmp_logiqa.txt
    return load_jsonl("tmp_logiqa.txt")

logiqa_train = download_logiqa(paths["train"])
logiqa_val   = download_logiqa(paths["val"])
logiqa_test  = download_logiqa(paths["test"])

# Convert to HF Dataset (CRITICAL FIX)
logiqa_train = Dataset.from_list(logiqa_train)
logiqa_val   = Dataset.from_list(logiqa_val)
logiqa_test  = Dataset.from_list(logiqa_test)

print(f"LogiQA train: {len(logiqa_train)}")
print(f"LogiQA val:   {len(logiqa_val)}")
print(f"LogiQA test:  {len(logiqa_test)}")


# ============================================================
# 9. PRINT DATASET SIZES
# ============================================================

print("\n===== ORIGINAL DATASET SIZES =====")
print(f"GSM8K:        {len(gsm8k['train'])}")
print(f"OpenOrca:    {len(orca['train'])}")
print(f"SciQ:        {len(sciq['train'])}")
print(f"CodeAlpaca:  {len(codealpaca['train'])}")
print(f"OpenHermes:  {len(openhermes['train'])}")

print("\n===== ADDITIONAL DATASETS =====")
print(f"Ultra subset:    {len(ultra_subset)}")
print(f"S1K:             {len(s1k_list)}")
print(f"Tulu-CoT:        {len(tulu_cot)}")
print(f"Reflection:      {len(reflection_list)}")
print(f"MBPP train:      {len(mbpp_train)}")
print(f"LogiQA train:    {len(logiqa_train)}")


# ============================================================
# 10. CREATE TRAINING SUBSETS
# ============================================================

gsm8k_small        = sample(gsm8k["train"], 600)
orca_small         = sample(orca["train"], 400)
sciq_small         = sample(sciq["train"], 300)
code_small         = sample(codealpaca["train"], 300)
openhermes_small   = openhermes["train"].select(range(500))
logiqa_small       = sample(logiqa_train, 400)

print("\n===== SAMPLED DATASETS =====")
print(f"GSM8K sampled:      {len(gsm8k_small)}")
print(f"OpenOrca sampled:  {len(orca_small)}")
print(f"SciQ sampled:      {len(sciq_small)}")
print(f"CodeAlpaca sampled:{len(code_small)}")
print(f"OpenHermes sampled:{len(openhermes_small)}")
print(f"LogiQA sampled:    {len(logiqa_small)}")

print("\n✅ ALL DATASETS READY FOR TRAINING")


### Data Preprocessing 

In [ ]:
import re
import json
import random
from tqdm import tqdm

# --------------------------
# SHARED HELPERS (updated & safer)
# --------------------------

# remove non-english unicode scripts often seen as noise
NON_ENGLISH_SCRIPTS = [
    r"[\u4e00-\u9fff]+",   # CJK
    r"[ぁ-んァ-ンー]+",     # Japanese hiragana/katakana
    r"[ก-๙]+",             # Thai
    r"[가-힣]+",            # Korean
    r"[அ-ஹ]+",            # Tamil
    r"[া-ৠ]+",            # Bengali/Assamese
    r"[\u0900-\u097F]+",   # Devanagari (Hindi, etc.)
    r"[\u0600-\u06FF]+",   # Arabic
]

def remove_non_english(text: str) -> str:
    if not text:
        return ""
    out = text
    for pat in NON_ENGLISH_SCRIPTS:
        out = re.sub(pat, " ", out)
    out = re.sub(r"\s{2,}", " ", out)
    return out.strip()

# Disallowed reasoning phrases filter
DISALLOWED_REASONING_PHRASES = [
    "let's think",
    "stream of consciousness",
    "my stream of consciousness",
    "my reasoning",
    "chain-of-thought",
    "chain of thought:",
]


def contains_disallowed_reasoning(text):
    t = (text or "").lower()
    return any(p in t for p in DISALLOWED_REASONING_PHRASES)

# Truncated answer check
def is_truncated(text):
    if not text:
        return True
    text = text.strip()

    # allow numeric, MCQ, and short factual answers
    if re.fullmatch(r"[A-Ea-e]|\d+(\.\d+)?", text):
        return False
    if len(text.split()) <= 3:
        return False

    return text[-1] not in ".!?"



def is_reasoning_too_similar(reasoning, answer, threshold=0.7):
    r = reasoning.lower()
    a = answer.lower()

    r_words = set(r.split())
    a_words = set(a.split())
    if not r_words or not a_words:
        return True

    overlap = len(r_words & a_words) / max(len(r_words), len(a_words))
    containment = a in r or r in a

    return overlap >= threshold or containment

def strip_xml(text):
    return re.sub(r"<[^>]+>", "", (text or "")).strip()

def remove_arrows(text):
    text = text or ""
    text = text.replace("=>", "=")
    text = re.sub(r"=\s*>", "=", text)
    text = re.sub(r"(?<=[=\$])\s*>", "", text)
    text = re.sub(r"\s*>\s*", " ", text)
    text = text.replace(">", "")
    text = re.sub(r"[=]{2,}", "=", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def trim_to_last_full_sentence(text):
    text = (text or "").strip()
    if not text:
        return text
    if text[-1] in ".!?":
        return text
    m = re.search(r'(?s)(.*[.!?])\s+[^.!?]*$', text)
    return m.group(1).strip() if m else text

def truncate_words_preserve_sentences(text, max_words=200):
    words = (text or "").strip().split()
    if len(words) <= max_words:
        return (text or "").strip()
    sentences = re.split(r'(?<=[.!?])\s+', (text or "").strip())
    out, count = "", 0
    for s in sentences:
        w = s.split()
        if count + len(w) <= max_words:
            out = (out + " " + s).strip()
            count += len(w)
        else:
            break
    return out if out else " ".join(words[:max_words]).strip()

def extract_numeric(text):
    if not text:
        return None
    nums = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    if not nums:
        return None
    return nums[-1].replace(",", "")

def extract_all_numeric(text):
    if not text:
        return []
    return [n.replace(",", "") for n in re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)]
# ============================================================
# 🔥 GLOBAL HARD FILTER (FINAL SAFETY LAYER)
# ============================================================

GLOBAL_DROP_PATTERNS = [
    r"<\|im_start\|>",
    r"<\|im_end\|>",
    r"stream of consciousness",
    r"rationale",
    r"come up with a question",
    r"provide a question",
    r"generate a question",
    r"the question and answer",
    r"rewrite the given sentence",
    r"ask the user",
    r"options:\s*-",
    r"\|\s*breed\s*\|",
    r"\|.*\|",  # tables
]

def is_clean_record(rec):
    if not rec:
        return False

    inp = (rec.get("input_text") or "").lower()
    tgt = (rec.get("target_text") or "").lower()

    # strict XML structure
    if not tgt.startswith("<reasoning>"):
        return False
    if "</reasoning>" not in tgt or "<answer>" not in tgt or "</answer>" not in tgt:
        return False
    if tgt.index("</reasoning>") > tgt.index("<answer>"):
        return False

    # global drop patterns
    for p in GLOBAL_DROP_PATTERNS:
        if re.search(p, inp) or re.search(p, tgt):
            return False

    # answer length constraint
    ans = re.search(r"<answer>(.*?)</answer>", tgt, re.S)
    if not ans:
        return False
    if len(ans.group(1).split()) > 25:
        return False

    return True

# ============================================================
# QUESTION DETECTION
# ============================================================

def is_list_or_pick_request(question):
    q = (question or "").lower()
    return bool(re.search(r'\b(give|list|name|pick|provide)\b.*\b(one|two|three|four|five|\d+)\b', q))

def is_multiple_choice(question):
    q = (question or "").lower()
    if "choose" in q or "options" in q:
        return True
    if re.search(r'\([a-e]\)', q): return True
    if re.search(r'\b(a|b|c|d|e)[\.\)]', q): return True
    return False

MATH_KEYWORDS = [
    "how many", "how much", "difference", "cost", "sum", "product", "volume",
    "profit", "distance", "time", "ratio", "percentage", "percent", "total",
    "perimeter", "area", "calculate", "find", "height", "width", "age",
    "split", "share", "each", "altogether", "equation", "remaining", "left", "leftover", "after"
]

def is_math_question(question):
    q = (question or "").lower()
    if is_list_or_pick_request(q): return False
    if any(k in q for k in MATH_KEYWORDS): return True
    if re.search(r'\d+\s*[*+\-/]\s*\d+', q): return True
    if re.search(r'\b(19|20)\d{2}\b', q): return False
    return False

# ============================================================
# normalize_answer (improved)
# ============================================================

def _first_numeric_preference_from_context(text, prefer_last=False, prefer_words=None):
    """Return best numeric candidate from text using contextual cues.
       prefer_last: if True prefer last numeric candidate.
       prefer_words: list of words to look for (e.g., ['remaining','left']) - prefer number near those words.
    """
    if not text:
        return None
    nums = extract_all_numeric(text)
    if not nums:
        return None
    if prefer_words:
        # find nearest numeric to any occurrence of prefer words
        text_l = text.lower()
        for w in prefer_words:
            idxs = [m.start() for m in re.finditer(re.escape(w), text_l)]
            if idxs:
                # for each occurrence, find closest number position
                for idx in idxs:
                    # find numbers and their spans
                    for m in re.finditer(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text):
                        nstart, nend = m.span()
                        # distance measure
                        dist = abs(nstart - idx)
                        # choose the number with minimum distance
                        # we simply return the first nearest
                        return m.group(0).replace(",", "")
    # prefer multi-digit numeric (likely final results)
    multi = [n for n in nums if len(re.sub(r'[-\.]','',n)) >= 2]
    if multi:
        return multi[-1] if prefer_last else multi[0]
    return nums[-1] if prefer_last else nums[0]

def normalize_answer(ans_text, question_text):
    if not ans_text and not question_text:
        return ""

    raw = (ans_text or "").strip()
    raw = remove_arrows(raw)
    raw = strip_xml(raw)
    # remove noisy non-english scripts early
    raw = remove_non_english(raw)

    q = (question_text or "").lower()

    # MULTIPLE CHOICE handling: prefer explicit A/B markers
    if is_multiple_choice(q):
        m = re.search(r"\b([A-Ea-e])\b", raw)
        if m:
            return m.group(1).upper()
        # fallback: look for "option a" or "A."
        m2 = re.search(r"\b(option|choice)\s+([a-e])\b", raw, flags=re.I)
        if m2:
            return m2.group(2).upper()
        first = raw.split()[0].strip().strip(".)(").upper() if raw.split() else ""
        if first in ["A","B","C","D","E"]:
            return first
        # otherwise return first sentence short
        return re.split(r'(?<=[.!?])\s+', raw)[0][:250].strip()

    # MATH handling: use context-aware numeric extraction
    if is_math_question(q):
        # if answer contains an explicit equals or boxed number - prefer that
        m_box = re.search(r"\\boxed\{\s*([-+]?\d+\.?\d*)\s*\}", raw)
        if m_box:
            return m_box.group(1)
        # equals patterns
        m_eq = re.search(r"(?:=\s*|equals\s+)([-+]?\d[\d,\.]*)", raw, flags=re.I)
        if m_eq:
            return m_eq.group(1).replace(",", "")
        # prefer numbers near words 'remaining', 'left', 'after' etc.
        prefer_words = ['remaining', 'remain', 'left', 'after', 'leftover']
        num_pref = _first_numeric_preference_from_context(raw, prefer_last=True, prefer_words=prefer_words)
        if num_pref:
            return num_pref
        # if none, prefer multi-digit last numeric (likely final)
        num_pref2 = _first_numeric_preference_from_context(raw, prefer_last=True)
        if num_pref2:
            return num_pref2
        # last resort: any numeric in raw
        any_num = extract_numeric(raw)
        if any_num:
            return any_num
        # if no numeric in answer text, look into question for numeric hints
        any_q_num = extract_numeric(q)
        return any_q_num or raw[:250]

    # For non-math: return first full sentence (cleaned), prefer short answers
    sentences = re.split(r'(?<=[.!?])\s+', raw)
    if sentences and sentences[0].strip():
        first = sentences[0].strip()
        # if first is just a digit and question not math, avoid returning numeric lists
        if re.fullmatch(r"\d+", first) and not is_math_question(q):
            # prefer a longer phrase if available
            if len(sentences) > 1 and len(sentences[1].split()) >= 3:
                return sentences[1].strip()
        return first[:250].strip()

    return raw.strip()
# ============================================================
# DERIVATIONAL ANSWER FILTER
# ============================================================

DERIVATION_WORDS = [
    "because", "therefore", "thus", "so ", "as a result",
    "which means", "this shows", "hence"
]

def is_derivational_answer(ans, reasoning):
    """Check if answer is explaining/deriving rather than giving final answer."""
    # Handle None or non-string types
    if not ans or not isinstance(ans, str):
        return False
    
    ans_l = ans.lower()
    # Contains derivation words
    if any(w in ans_l for w in DERIVATION_WORDS):
        # allow short explanations (<= 12 words)
        if len(ans_l.split()) <= 12:
            return False
        return True

    
    # Overlap check (answer must be mostly new tokens)
    r_words = set(reasoning.lower().split())
    a_words = set(ans_l.split())
    if a_words:
        overlap = len(r_words & a_words) / len(a_words)
        if overlap > 0.4:
            return True
    
    return False
# ============================================================
# TARGET TEXT BUILDER (safe)
# ============================================================

def make_target_text(reasoning, answer):
    reasoning = (reasoning or "").strip()
    answer = (answer or "").strip()

    # HARD DROP if reasoning or answer is weak
    if len(reasoning.split()) < 15:
        return None
    if not answer:
        return None

    # ❌ NEW: Drop answers that explain or derive
    if is_derivational_answer(answer, reasoning):
        return None

    reasoning = remove_arrows(reasoning)
    reasoning = remove_non_english(reasoning)
    reasoning = truncate_words_preserve_sentences(reasoning, 220)
    reasoning = trim_to_last_full_sentence(reasoning)

    answer = strip_xml(answer)
    answer = remove_non_english(answer)
    answer = answer.strip()

    return (
        "<reasoning>\n"
        f"{reasoning}\n"
        "</reasoning>\n"
        "<answer>\n"
        f"{answer}\n"
        "</answer>"
    )


def make_record(prompt, reasoning, answer):
    target = make_target_text(reasoning, answer)
    if target is None:
        return None

    return {
        "input_text": f"Question: {prompt.strip()}",
        "target_text": target
    }

# ============================================================
# DATASET-SPECIFIC FORMATTERS (safe versions)
# ============================================================

def extract_reasoning_and_answer(target_text):
    """Split <reasoning>...</reasoning> and <answer>...</answer> safely."""
    reasoning = re.search(r"<reasoning>(.*?)</reasoning>", target_text, re.DOTALL)
    answer = re.search(r"<answer>(.*?)</answer>", target_text, re.DOTALL)

    reasoning = reasoning.group(1).strip() if reasoning else "<no reasoning>"
    answer = answer.group(1).strip() if answer else "<no answer>"

    return reasoning, answer

# ------------------------------------------------------------
# FORMAT ULTRAFEEDBACK (safe)
# ------------------------------------------------------------

print("\nFormatting UltraFeedback...")
formatted_ultra = []

def extract_uf_qa(item):
    """Extract question + answer from UltraFeedback in a safe way."""
    if "instruction" in item:
        q = item["instruction"]
    elif "question" in item:
        q = item["question"]
    elif "prompt" in item:
        q = item["prompt"]
    else:
        q = "(Unknown Question Format)"

    if "response" in item:
        ans = item["response"]
    elif "answer" in item:
        ans = item["answer"]
    elif "completions" in item and item["completions"]:
        ans = item["completions"][0].get("response", "")
    else:
        ans = ""

    return q or "", ans or ""

for item in tqdm(ultra_subset[:50]):
    q, raw_answer = extract_uf_qa(item)
    q = (q or "").strip()

    # Normalize and clean
    raw_clean = strip_xml(remove_arrows(raw_answer))
    raw_clean = remove_non_english(raw_clean)
    raw_clean = truncate_words_preserve_sentences(raw_clean, 120)
    ans = normalize_answer(raw_clean, q)

    # Repair extremely short answers (<4 words)
    if len(ans.split()) < 4:
        full_sents = re.split(r'(?<=[.!?])\s+', raw_clean)
        if full_sents and len(full_sents[0].split()) >= 4:
            ans = full_sents[0].strip()
        else:
            fallback = " ".join(raw_clean.split()[:18])
            if len(fallback.split()) >= 4:
                ans = fallback

    # Reasoning extraction
    reasoning_raw = strip_xml(raw_answer)
    reasoning_raw = remove_non_english(reasoning_raw)
    reasoning_raw = remove_arrows(reasoning_raw)
    reasoning_raw = truncate_words_preserve_sentences(reasoning_raw, 90)
    reasoning_raw = trim_to_last_full_sentence(reasoning_raw)

    if (
        len(reasoning_raw.split()) < 15
        or reasoning_raw.strip().lower() == ans.strip().lower()
    ):
        continue

    # Add new filters
    if contains_disallowed_reasoning(reasoning_raw):
        continue
    if is_derivational_answer(ans, reasoning_raw):
        continue
    if is_truncated(ans):
        continue

    rec = make_record(q, reasoning_raw, ans)
    if rec:
        formatted_ultra.append(rec)


print("UltraFeedback formatted count:", len(formatted_ultra))

# ------------------------------------------------------------
# FORMAT MBPP (code reasoning — FINAL & FIXED)
# ------------------------------------------------------------

formatted_mbpp = []
print("\nFormatting MBPP (fixed code blocks)…")


# ------------------------------------------------------------
# FORMAT MBPP (FINAL — SAFE VERSION)
# ------------------------------------------------------------

formatted_mbpp = []
print("\nFormatting MBPP (safe, no reconstruction)…")

def build_mbpp_reasoning(text):
    return (
        "The problem can be solved by analyzing the input conditions and applying "
        "the required logical checks to produce the correct result while handling "
        "edge cases appropriately."
    )

for item in tqdm(mbpp_train):
    prompt = (item.get("text") or "").strip()
    code = (item.get("code") or "").strip()

    if not prompt or not code:
        continue

    # Safety
    if contains_disallowed_reasoning(prompt):
        continue

    reasoning = build_mbpp_reasoning(prompt)

    # KEEP CODE AS-IS (do NOT touch formatting)
    answer = strip_xml(code).strip()

    # Minimal sanity check: must look like a function
    if not answer.startswith("def "):
        continue

    rec = make_record(prompt, reasoning, answer)
    if rec:
        formatted_mbpp.append(rec)

print("Formatted MBPP samples:", len(formatted_mbpp))


# ------------------------------------------------------------
# FORMAT LOGIQA 2.0 (logical reasoning — strict)
# ------------------------------------------------------------

formatted_logiqa = []
print("\nFormatting LogiQA 2.0…")

OPTION_LABELS = ["A", "B", "C", "D", "E"]

def build_logiqa_prompt(item):
    context = (item.get("text") or "").strip()
    question = (item.get("question") or "").strip()
    options = item.get("options") or []

    if not context or not question or len(options) < 2:
        return None

    context = remove_non_english(context)
    question = remove_non_english(question)

    opt_lines = []
    for i, opt in enumerate(options):
        if i >= len(OPTION_LABELS):
            break
        opt_clean = remove_non_english(opt)
        opt_lines.append(f"{OPTION_LABELS[i]}. {opt_clean}")

    return (
        f"{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Options:\n" + "\n".join(opt_lines)
    )

def build_logiqa_reasoning(context, question):
    return (
        "The correct answer can be determined by carefully analyzing the given passage, "
        "identifying the relevant conditions stated, and logically evaluating which option "
        "is fully supported by the information without introducing assumptions."
    )

for item in tqdm(logiqa_train):
    options = item.get("options") or []
    answer_idx = item.get("answer")

    if not isinstance(answer_idx, int):
        continue
    if answer_idx < 0 or answer_idx >= len(options):
        continue

    prompt = build_logiqa_prompt(item)
    if not prompt:
        continue

    reasoning = build_logiqa_reasoning(
        item.get("text", ""),
        item.get("question", "")
    )

    answer = remove_non_english(options[answer_idx]).strip()

    # Reject trivial or malformed answers
    if len(answer.split()) < 3:
        continue

    # Safety filters
    if contains_disallowed_reasoning(reasoning):
        continue
    if is_truncated(answer):
        continue

    rec = make_record(prompt, reasoning, answer)
    if rec:
        formatted_logiqa.append(rec)

print("Formatted LogiQA samples:", len(formatted_logiqa))


# ------------------------------------------------------------
# FORMAT S1K (safe)
# ------------------------------------------------------------

def clean_s1k_reasoning(text):
    if not text:
        return None

    t = strip_xml(remove_arrows(str(text))).strip()
    t = remove_non_english(t)
    cleaned = truncate_words_preserve_sentences(t, 120)
    cleaned = trim_to_last_full_sentence(cleaned)

    if len(cleaned.split()) < 15:
        return None

    return cleaned


formatted_s1k = []
print("Formatting S1K…")

for _, row in tqdm(df_s1k.iterrows(), total=len(df_s1k)):
    q = str(row.get("question", "")).strip()
    raw_reasoning = row.get("gemini_thinking_trajectory", "")
    raw_answer = row.get("solution", "")

    reasoning = clean_s1k_reasoning(raw_reasoning)
    if reasoning is None:
        continue

    answer = strip_xml(raw_answer).strip()
    if not answer:
        continue

    # Add new filters with correct variable names
    if contains_disallowed_reasoning(reasoning):
        continue
    if is_derivational_answer(answer, reasoning):
        continue
    if is_truncated(answer):
        continue

    rec = make_record(q, reasoning, answer)
    if rec:
        formatted_s1k.append(rec)


print("Formatted S1K count:", len(formatted_s1k))


# ------------------------------------------------------------
# FORMAT TULU COT (safe)
# ------------------------------------------------------------

def extract_tulu_fields(item):
    msgs = item.get("messages", [])
    if len(msgs) < 2:
        return "(Missing Question)", "(Missing Answer)"
    q = msgs[0].get("content", "").strip()
    a = msgs[1].get("content", "").strip()
    return q, a

def split_reasoning_and_answer(text):
    text = text.strip()
    m = re.search(r"(?:final answer is|answer is)\s+(.+)", text, re.I)
    if m:
        answer = m.group(1).strip()
        reasoning = text[:m.start()].strip()
        return reasoning, answer
    sentences = re.split(r"(?<=[\.\?\!])\s+", text)
    if len(sentences) > 1:
        answer = sentences[-1].strip()
        reasoning = " ".join(sentences[:-1]).strip()
        return reasoning, answer
    return text, text

formatted_tulu = []
print("Formatting Tulu-CoT…")

for item in tqdm(tulu_cot[:3000]):
    q, full_answer = extract_tulu_fields(item)
    q = remove_non_english(q)
    full_answer = remove_non_english(full_answer)

    reasoning, ans = split_reasoning_and_answer(full_answer)
    reasoning = strip_xml(remove_arrows(reasoning))
    reasoning = truncate_words_preserve_sentences(reasoning, 120)
    reasoning = trim_to_last_full_sentence(reasoning)
    if len(reasoning) < 15:
        continue

    ans = strip_xml(ans).strip()

    # Add new filters with correct variable names
    if contains_disallowed_reasoning(reasoning):
        continue
    if is_derivational_answer(ans, reasoning):
        continue
    if is_truncated(ans):
        continue

    rec = make_record(q, reasoning, ans)
    if rec:
        formatted_tulu.append(rec)


print("Formatted Tulu-CoT samples:", len(formatted_tulu))

# ------------------------------------------------------------
# FORMAT REFLECTION-QWEN (safe)
# ------------------------------------------------------------

def clean_qwen_tags(text):
    if not text:
        return ""
    text = re.sub(r"<\|im_start\|>[^ \n]*", "", text)
    text = re.sub(r"<\|im_end\|>", "", text)
    return text.strip()

def extract_reflection_fields(raw_item):
    txt = clean_qwen_tags(raw_item.get("text", ""))

    user_blocks = re.findall(
        r"<\|im_start\|>user\s*(.*?)\s*(?=<\|im_start\|>|$)",
        raw_item.get("text", ""),
        flags=re.S
    )
    q = user_blocks[0].strip() if user_blocks else None
    if not q:
        return None

    reflections = re.findall(r"<reflection>\s*(.*?)\s*</reflection>", txt, flags=re.S)
    if not reflections:
        return None

    reasoning = reflections[0].strip()
    reasoning = strip_xml(remove_arrows(reasoning))
    reasoning = remove_non_english(reasoning)
    reasoning = truncate_words_preserve_sentences(reasoning, 160)
    reasoning = trim_to_last_full_sentence(reasoning)

    if len(reasoning.split()) < 15:
        return None

    m = re.search(r"<output>\s*(.*?)\s*</output>", txt, flags=re.S)
    if not m:
        return None

    ans = strip_xml(m.group(1)).strip()
    if not ans:
        return None

    return q, reasoning, ans

reflection_subset = random.sample(reflection_list, min(3000, len(reflection_list)))
formatted_reflection = []
print("\nFormatting Reflection-Qwen…")
for item in tqdm(reflection_subset):
    res = extract_reflection_fields(item)
    if res is None:
        continue

    q, reasoning_raw, ans = res
    q = remove_non_english(q)

    # Add new filters
    if contains_disallowed_reasoning(reasoning_raw):
        continue
    if is_derivational_answer(ans, reasoning_raw):
        continue
    if is_truncated(ans):
        continue

    rec = make_record(q, reasoning_raw, ans)
    if rec:
        formatted_reflection.append(rec)

print("Reflection-Qwen formatted:", len(formatted_reflection))

# ------------------------------------------------------------
# Other datasets: GSM8K, OpenHermes, SciQ, CodeAlpaca (safe)
# ------------------------------------------------------------

cleaned_records = []

# ---------- 1. FORMAT GSM8K ----------
formatted_gsm8k = []
print("\nFormatting GSM8K…")
for item in tqdm(gsm8k_small):
    q = (item.get("question") or "").strip()
    if not q:
        continue
    full = (item.get("answer") or "").strip()
    if not full:
        continue

    # reasoning may include solution steps and final answer split by ####
    reasoning_raw = full.split("####")[0].strip() if "####" in full else full.strip()
    reasoning_raw = strip_xml(remove_arrows(reasoning_raw))
    reasoning_raw = remove_non_english(reasoning_raw)
    reasoning_raw = truncate_words_preserve_sentences(reasoning_raw, 140)
    # preserve math operators spacing carefully
    reasoning_raw = re.sub(r"(?<=\d)\*(?=\d)", " * ", reasoning_raw)
    reasoning_raw = re.sub(r"(?<=\d)/(?=\d)", " / ", reasoning_raw)
    reasoning_raw = re.sub(r"(?<=\d)\+(?=\d)", " + ", reasoning_raw)
    reasoning_raw = re.sub(r"(?<=\d)-(?=\d)", " - ", reasoning_raw)
    reasoning_raw = trim_to_last_full_sentence(reasoning_raw)

    # gold extraction: prefer explicit golden after ####, otherwise try to extract numeric sensibly
    if "####" in full:
        gold = full.split("####")[-1].strip()
    else:
        gold = full.strip()
    ans = normalize_answer(gold, q)

    if reasoning_raw.strip().lower() == (ans or "").strip().lower() or len(reasoning_raw) < 15:
        continue
    if is_derivational_answer(ans, reasoning_raw):
        continue

    rec = make_record(q, reasoning_raw, ans)
    if rec:
        formatted_gsm8k.append(rec)

cleaned_records.extend(formatted_gsm8k)

# ---------- 2. FORMAT OpenHermes ----------
print("\nSkipping OpenHermes — no genuine reasoning.")
# OpenHermes excluded as requested

# ---------- 3. FORMAT SciQ ----------
formatted_sciq = []
print("\nFormatting SciQ…")

for item in tqdm(sciq_small):
    q = (item.get("question") or "").strip()
    support = (item.get("support") or "").strip()
    ans_raw = (item.get("correct_answer") or "").strip()

    if not q or not support or not ans_raw:
        continue

    reasoning = strip_xml(remove_arrows(support))
    reasoning = remove_non_english(reasoning)
    reasoning = truncate_words_preserve_sentences(reasoning, 120)
    reasoning = trim_to_last_full_sentence(reasoning)

    if len(reasoning.split()) < 15:
        continue

    answer = strip_xml(ans_raw).strip()

    rec = make_record(q, reasoning, answer)
    if rec:
        formatted_sciq.append(rec)

cleaned_records.extend(formatted_sciq)



# ============================================================
# SHOW 2 SAMPLES FROM EACH DATASET (safety preview)
# ============================================================

def extract_reasoning_and_answer(target_text):
    reasoning = re.search(r"<reasoning>(.*?)</reasoning>", target_text, re.DOTALL)
    answer = re.search(r"<answer>(.*?)</answer>", target_text, re.DOTALL)
    reasoning = reasoning.group(1).strip() if reasoning else "<no reasoning>"
    answer = answer.group(1).strip() if answer else "<no answer>"
    return reasoning, answer

print("\n" + "="*60)
print("SHOWING 2 SAMPLES FROM EACH DATASET (preview)")
print("="*60)

# select two from each list if available
def show_samples(name, lst):
    print(f"\n\n=========== {name.upper()} SAMPLES ===========")
    for i, rec in enumerate(lst[:2], start=1):
        print(f"\n========== SAMPLE {i} ==========\n")
        question = rec["input_text"]
        reasoning, final = extract_reasoning_and_answer(rec["target_text"])
        print("QUESTION:\n", question, "\n")
        print("REASONING:\n", reasoning, "\n")
        print("FINAL ANSWER:\n", final, "\n")

show_samples("UltraFeedback", formatted_ultra)
show_samples("S1K", formatted_s1k)
show_samples("Tulu-CoT", formatted_tulu)
show_samples("Reflection-Qwen", formatted_reflection)
show_samples("Other datasets", cleaned_records[:2])

# NEW
show_samples("MBPP", formatted_mbpp)
show_samples("LogiQA 2.0", formatted_logiqa)


# ============================================================
# COMBINE ALL DATASETS INTO UNIFIED DATASET (SHUFFLED)
# ============================================================

print("\n" + "="*60)
print("COMBINING ALL DATASETS INTO UNIFIED DATASET")
print("="*60)

all_formatted_datasets = []
all_formatted_datasets.extend(formatted_ultra)
all_formatted_datasets.extend(formatted_s1k)
all_formatted_datasets.extend(formatted_tulu)
all_formatted_datasets.extend(formatted_reflection)
all_formatted_datasets.extend(cleaned_records)
all_formatted_datasets.extend(formatted_mbpp)
all_formatted_datasets.extend(formatted_logiqa)


# Shuffle to avoid long contiguous blocks from one source
random.shuffle(all_formatted_datasets)

print(f"\n=== DATASET COUNTS ===")
print(f"UltraFeedback: {len(formatted_ultra)}")
print(f"S1K: {len(formatted_s1k)}")
print(f"MBPP: {len(formatted_mbpp)}")
print(f"LogiQA: {len(formatted_logiqa)}")

print(f"Tulu-CoT: {len(formatted_tulu)}")
print(f"Reflection-Qwen: {len(formatted_reflection)}")
print(f"Other datasets (GSM8K, SciQ): {len(cleaned_records)}")
print(f"TOTAL UNIFIED DATASET: {len(all_formatted_datasets)}")

# ============================================================
# SAVE UNIFIED DATASET (FINAL CLEAN PASS)
# ============================================================

clean_final = [r for r in all_formatted_datasets if is_clean_record(r)]

print("\n=== FINAL CLEAN DATASET ===")
print("Before filtering:", len(all_formatted_datasets))
print("After filtering :", len(clean_final))

output_filename = "unified_training_data.jsonl"
print(f"\nSaving unified dataset to: {output_filename}")

with open(output_filename, "w", encoding="utf-8") as f:
    for record in clean_final:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"✅ Successfully saved {len(clean_final)} clean records to {output_filename}")


# ============================================================
# SHOW FINAL RANDOM SAMPLES FROM UNIFIED DATASET
# ============================================================

print("\n" + "="*60)
print("FINAL RANDOM SAMPLES FROM UNIFIED DATASET")
print("="*60)

if all_formatted_datasets:
    random_samples = random.sample(clean_final, min(3, len(clean_final)))
    for i, rec in enumerate(random_samples, start=1):
        print(f"\n===== RANDOM SAMPLE {i} =====")
        print("INPUT:")
        print(rec["input_text"])
        reasoning, answer = extract_reasoning_and_answer(rec["target_text"])
        print("\nREASONING:")
        print(reasoning)
        print("\nFINAL ANSWER:")
        print(answer)
        print("\n" + "="*40)

print(f"\n🎯 UNIFIED DATASET COMPLETE!")
print(f"📁 File: {output_filename}")
print(f"📊 Total records: {len(all_formatted_datasets)}")

In [ ]:
import json
import random

file_path = "unified_training_data.jsonl"

# Load dataset into memory
records = [json.loads(line) for line in open(file_path, "r", encoding="utf-8")]

# Print total count
print("TOTAL RECORDS IN DATASET:", len(records))


# Pick 5 random samples
samples = random.sample(records, 5)

for i, rec in enumerate(samples):
    print(f"\n===== SAMPLE {i+1} =====")
    print("INPUT:")
    print(rec["input_text"])
    print("\nTARGET:")
    print(rec["target_text"])
    print("\n========================")
